In [1]:
import pandas as pd
import json
import ast
import os
import re

In [2]:
def validate_property(properties): 
    for prop in properties:  
        if not isinstance(prop, dict):  
            return False, "Each item in 'PROPERTY' must be a dictionary"  
        if 'property_name' not in prop or 'modifier' not in prop or 'property_type' not in prop:  
            return False, "Each property dictionary must contain 'property_name', 'modifier', and 'property_type' keys"  
        if not isinstance(prop['modifier'], dict):  
            return False, "'modifier' must be a dictionary"  
        for key in ['value', 'min', 'max', 'unit']:  
            if key not in prop['modifier']:  
                return False, f"Missing key '{key}' in modifier"  
    return True, "" 

def validate_filler(fillers): 
    for filler in fillers:  
        if not isinstance(filler, dict):  
            return False, "Each item in 'FILLER' must be a dictionary"  
        if 'filler_name' in filler:  
            if not isinstance(filler['filler_name'], list):  
                return False, "'filler_name' must be a list"  
        elif 'total_load' in filler:  
            if not isinstance(filler['total_load'], dict):  
                return False, "'total_load' must be a dictionary"  
            for key in ['value', 'min', 'max']:  
                if key not in filler['total_load']:  
                    return False, f"Missing key '{key}' in total_load"  
        else:  
            return False, "Each filler dictionary must contain either 'filler_name' or 'total_load'" 
    return True, "" 


def validate_auto_cert(auto_certs):  
    for cert in auto_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'AUTO_CERT' must be a dictionary"  
        if 'oem' not in cert or 'certs' not in cert:  
            return False, "Each auto cert dictionary must contain 'oem' and 'certs'"  
        if not isinstance(cert['certs'], list):  
            return False, "'certs' must be a list"  
    return True, ""  
  
def validate_railway_cert(railway_certs):  
    for cert in railway_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'RAILWAY_CERT' must be a dictionary"  
        if 'standard' not in cert or 'hazard_level' not in cert or 'req_set' not in cert:  
            return False, "Each railway cert dictionary must contain 'standard', 'hazard_level', and 'req_set'"  
        if not isinstance(cert['hazard_level'], list) or not isinstance(cert['req_set'], list):  
            return False, "'hazard_level' and 'req_set' must be lists"  
    return True, ""  
  
def validate_water_cert(water_certs):  
    for cert in water_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'WATER_CERT' must be a dictionary"  
        if 'standard' not in cert or 'temp' not in cert:  
            return False, "Each water cert dictionary must contain 'standard' and 'temp'"  
        if not isinstance(cert['temp'], list):  
            return False, "'temp' must be a list"  
    return True, ""  
  
def validate_nsf_cert(nsf_certs):  
    if not isinstance(nsf_certs, list):  
        return False, "'NSF_CERT' must be a list"  
    return True, ""  
  
def validate_dict(data):   
    expected_format = {  
    'GRADE': list,  
    'APPLICATION': list,  
    'BRAND': list,  
    'POLYMER': list,  
    'PROPERTY': list,  
    'FILLER': list,  
    'FEATURE': list,  
    'PROCESSING': list,  
    'DELIVERY_FORM': list,  
    'COMPETITOR_GRADE': list,  
    'AUTO_CERT': list,  
    'RAILWAY_CERT': list,  
    'WATER_CERT': list,  
    'NSF_CERT': list,  
    'INDUSTRY': list,  
    'REGION': list  
    }  
    
     
    # Find any entities in the input data that are not expected  
    unknown_entities = set(data.keys() - expected_format.keys())
      
    # if unknown_entities and not "CERTIFICATION" in unknown_entities:  
    if unknown_entities:  
        return False, f"Unknown entities detected: {', '.join(unknown_entities)}" 
    
#     if "CERTIFICATION" in unknown_entities and data["CERTIFICATION"]:
# # #     if "CERTIFICATION" in unknown_entities:
#         return False, f"Value found in CERTIFICATION" 
  
    # Check if all required keys are present and of the correct type  
    for key, expected_type in expected_format.items():  
        if key not in data:  
            return False, f"Missing key: {key}"  
        if not isinstance(data[key], expected_type):  
            return False, f"Key '{key}' is not of type {expected_type.__name__}"   
    
    # Validate PROPERTY  
    valid, message = validate_property(data['PROPERTY'])  
    if not valid:  
        return False, message
    
    # Validate FILLER  
    valid, message = validate_filler(data['FILLER'])  
    if not valid:  
        return False, message

  
    # Validate AUTO_CERT  
    valid, message = validate_auto_cert(data['AUTO_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate RAILWAY_CERT  
    valid, message = validate_railway_cert(data['RAILWAY_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate WATER_CERT  
    valid, message = validate_water_cert(data['WATER_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate NSF_CERT  
    valid, message = validate_nsf_cert(data['NSF_CERT'])  
    if not valid:  
        return False, message  
  
    return True, "Validation successful"  

In [3]:
data = {  
    'GRADE': ['Celcon m90'],  
    'APPLICATION': ['Speaker Grill'],  
    'BRAND': ['celcon', 'hostaform'],  
    'POLYMER': ['pps', 'pom'],  
    'PROPERTY': [  
        {'property_name': 'density',  
         'modifier': {'value': 3000, 'min': 2700, 'max': 3300, 'unit': None},  
         'property_type': 'property'},
        {'property_name': 'density',  
         'modifier': {'value': 110.70000000000000, 'min': '110.700000000000000', 'max': '135.3', 'unit': None},  
         'property_type': 'property'},
        {'property_name': 'density',  
         'modifier': {'value': '2.2e-04', 'min': 2.2e-04, 'max': '2.2e+04', 'unit': None},  # cant hanlde for float type (2.2e-04)
         'property_type': 'property'},
        {'property_name': 'minimum thickness (mm)',  
         'modifier': {'value': 0.4, 'min': 0.4, 'max': 0.4, 'unit': 'mm'},  
         'property_type': 'ul_property'},  
        {'property_name': 'flame  rating',  
         'modifier': {'value': 'v0', 'min': None, 'max': '', 'unit': ''},  
         'property_type': 'ul_sub_property'}  
    ],  
    'FILLER': [{'filler_name': ['glass   fiber', 'mineral']},  
               {'total_load': {'value': 30, 'min': ' ', 'max': 40}}],  
    'FEATURE': ['heat resistant', 'impact modified'],  
    'PROCESSING': ['coatable', 'blow moulding'],  
    'DELIVERY_FORM': ['granules'],  
    'COMPETITOR_GRADE': [],  
    'AUTO_CERT': [{'OEM': 'FORD', 'CERTS': ['WSS-M4D1038', 'WSS-M9P14-A1']},  
                  {'OEM': 'BMW', 'CERTS': ['MAD P1 0367']}],  
    'RAILWAY_CERT': [{'Standard': 'EN45545-2',  
                      'Hazard_Level': ['H1', 'H2'],  
                      'Req_Set': ['R22', 'R26']}],  
    'WATER_CERT': [{'Standard': 'ACS', 'Temp': [60, 100]},  
                   {'Standard': 'NSF 61', 'Temp': [100]}],  
    'NSF_CERT': ['NSF 61', '  ', 'NSF  42'], 
    'INDUSTRY': ['INDUSTRIAL'], 
    'REGION': ['AMERICAS'] 
}  
  
def process_value(value, field):
    if field == 'PROPERTY':
        if isinstance(value, int) or isinstance(value, float):  
            return str(float(value))  
        elif isinstance(value, str):
            try: 
                if 'e-' in value or 'e+' in value:
                    pass
                else:
                    value = str(float(value))
            except:
                pass
                
            return re.sub("(\s+)", " ", value.lower().strip())
    else:
        if isinstance(value, int) or isinstance(value, float):  
            return str(value)  
        elif isinstance(value, str):               
            return re.sub("(\s+)", " ", value.lower().strip())   
            
    return value  
  
def format_output(d, is_nested=False):  
    new_dict = {}  
    for k, v in d.items():  
        new_key = k.lower() if is_nested else k.upper()  
        if isinstance(v, dict):  
            new_dict[new_key] = format_output(v, is_nested=True)  
        elif isinstance(v, list):  
            new_list = []  
            for item in v:  
                if isinstance(item, dict):  
                    new_list.append(format_output(item, is_nested=True))  
                else:
                    if process_value(item, k): 
                        new_list.append(process_value(item, k))
                    else:
                        if new_key == 'unit':
                            new_list.append('')
                        elif new_key in ['value', 'min', 'max']:
                            new_list.append(None)
                        
            new_dict[new_key] = new_list  
        else:
            f = k
            if 'unit' in d:
                f = 'PROPERTY'
                
            if process_value(v, f):
                new_dict[new_key] = process_value(v, f)  
            else:
                if new_key == 'unit':
                    new_dict[new_key] = ''
                elif new_key in ['value', 'min', 'max']:
                    new_dict[new_key] = None
                    
    return new_dict  
  
formated_output = format_output(data)  
formated_output 

{'GRADE': ['celcon m90'],
 'APPLICATION': ['speaker grill'],
 'BRAND': ['celcon', 'hostaform'],
 'POLYMER': ['pps', 'pom'],
 'PROPERTY': [{'property_name': 'density',
   'modifier': {'value': '3000.0',
    'min': '2700.0',
    'max': '3300.0',
    'unit': ''},
   'property_type': 'property'},
  {'property_name': 'density',
   'modifier': {'value': '110.7', 'min': '110.7', 'max': '135.3', 'unit': ''},
   'property_type': 'property'},
  {'property_name': 'density',
   'modifier': {'value': '2.2e-04',
    'min': '0.00022',
    'max': '2.2e+04',
    'unit': ''},
   'property_type': 'property'},
  {'property_name': 'minimum thickness (mm)',
   'modifier': {'value': '0.4', 'min': '0.4', 'max': '0.4', 'unit': 'mm'},
   'property_type': 'ul_property'},
  {'property_name': 'flame rating',
   'modifier': {'value': 'v0', 'min': None, 'max': None, 'unit': ''},
   'property_type': 'ul_sub_property'}],
 'FILLER': [{'filler_name': ['glass fiber', 'mineral']},
  {'total_load': {'value': '30', 'min': N

In [4]:
file_path = "data/Training_Data_27_01_25/without_formatting/Training_Data_27_01_25.xlsx"
file_name = os.path.splitext(os.path.basename(file_path))[0]
folder_path = os.path.splitext(file_path)[0]

In [5]:
df = pd.read_excel(file_path)

outputs = []
for idx, q, output in zip(df['Sl. No.'], df['Query'], df["Output"]):
    if not isinstance(output, dict):  
        output = ast.literal_eval(output)

    formated_output = format_output(output)
    outputs.append(formated_output)
    
df["Output"] = outputs
df["Query"] = df["Query"].astype(str).str.lower()
df

,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5
...,...,...,...,...
61527,61528,what is the glass transition of fp 6e5901a80?,"{'GRADE': ['fp 6e5901a80'], 'APPLICATION': [],...",prod_generated_data_prop_grade_27_01_25_46
61528,61529,elongational stress f of lfrt cfr-tp pet gf60-10,"{'GRADE': ['lfrt cfr-tp pet gf60-10'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_47
61529,61530,tensile creep modulus at 1h of frianyl a3 v2 o...,"{'GRADE': ['frianyl a3 v2 or 2003/p'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_48
61530,61531,flexural stress at 3.5% iso 178 (mpa) of impet...,"{'GRADE': ['impet 830r'], 'APPLICATION': [], '...",prod_generated_data_prop_grade_27_01_25_49


In [6]:
incorrect = []
for idx, q, output in zip(df['Sl. No.'], df['Query'], df["Output"]):
    try:
        if not isinstance(output, dict):  
            output = ast.literal_eval(output)
    except Exception as e:
        print(idx, e)
        print("\n", q, "\n")
        print(output)
        incorrect.append(idx)
        continue
        
    is_valid, message = validate_dict(output)

    for k in output:
        if "" in output[k]:
            print(True)
    
    if not is_valid:
        print(idx, message)
        print("\n", q, "\n")
        print(output, "\n\n")
        incorrect.append(idx)

print(len(incorrect))

0


In [7]:
processing_syn_mapping = {
    'blow molding': 'blow molding',
    'blowmolding': 'blow molding',
    'bm': 'blow molding',
    'blow moulding': 'blow molding',
    'blow moldable': 'blow molding',
    'blow molded': 'blow molding',
    'calendering': 'calendering',
    'calandering': 'calendering',
    'calendered': 'calendering',
    'calendr': 'calendering',
    'calenderable': 'calendering',
    'casting': 'casting',
    'castable': 'casting',
    'coatable': 'coatable',
    'coating': 'coatable',
    'polymeric coating': 'coatable',
    'polymeric': 'coatable',
    'spin coating': 'coatable',
    'coextrusion': 'coextrusion',
    'compression molding': 'compression molding',
    'compression moulding': 'compression molding',
    'matched-die moulding': 'compression molding',
    'matched-die molding': 'compression molding',
    'matched die molding': 'compression molding',
    'matched die moulding': 'compression molding',
    'cold compression': 'compression molding',
    'hot compression': 'compression molding',
    'compmold': 'compression molding',
    'compression moldable': 'compression molding',
    'compression mouldable': 'compression molding',
    'extrusion - hose': 'extrusion - hose',
    'hose extrusion': 'extrusion - hose',
    'extrusion - ram': 'ram extrusion',
    'ram extrusion': 'ram extrusion',
    'extrusion - small tubing': 'extrusion - small tubing',
    'extrusion - wire and cable': 'extrusion - wire and cable',
    'cable extrusion': 'extrusion - wire and cable',
    'wire extrusion': 'extrusion - wire and cable',
    'extrusion wire coating': 'extrusion - wire and cable',
    'extrusion cable coating': 'extrusion - wire and cable',
    'wire coating': 'extrusion - wire and cable',
    'cable coating': 'extrusion - wire and cable',
    'extrusion blow molding': 'extrusion blow molding',
    'extrusion blow moulding': 'extrusion blow molding',
    'extrusion blow moldable': 'extrusion blow molding',
    'extrusion blow mouldable': 'extrusion blow molding',
    'fiber spinning / gel spinning': 'fiber spinning / gel spinning',
    'fibre spinning / gel spinning': 'fiber spinning / gel spinning',
    'fibre spinning / gel extrusion': 'fiber spinning / gel spinning',
    'fiber spinning / gel extrusion': 'fiber spinning / gel spinning',
    'fiber spinning': 'fiber spinning / gel spinning',
    'fibre spinning': 'fiber spinning / gel spinning',
    'semi melt spinning': 'fiber spinning / gel spinning',
    'gel-electrospinning': 'fiber spinning / gel spinning',
    'gel electrospinning': 'fiber spinning / gel spinning',
    'gel spinning': 'fiber spinning / gel spinning',
    'film extrusion': 'film extrusion',
    'cast film extrusion': 'film extrusion',
    'foam processing': 'foam processing',
    'foam extrusion': 'foam processing',
    'foamable': 'foam processing',
    'bead foam extrusion': 'foam processing',
    'physical foaming': 'foam processing',
    'foaming': 'foam processing',
    'mucell': 'foam processing',
    'injection blow molding': 'injection blow molding',
    'injection blow moulding': 'injection blow molding',
    'injection blow moldable': 'injection blow molding',
    'ibm': 'injection blow molding',
    'injection molding': 'injection molding',
    'injection moldable': 'injection molding',
    'injection moulding': 'injection molding',
    'injmold': 'injection molding',
    'injmould': 'injection molding',
    'injection': 'injection molding',
    'injection mouldable': 'injection molding',
    'melt blowing / nonwovens': 'melt blowing / nonwovens',
    'melt blown': 'melt blowing / nonwovens',
    'multi injection molding': 'multi injection molding',
    'multi injection moulding': 'multi injection molding',
    'multi-material injection molding': 'multi injection molding',
    'multi material injection molding': 'multi injection molding',
    'multi-material injection moulding': 'multi injection molding',
    'multi material injection moulding': 'multi injection molding',
    'mmm': 'multi injection molding',
    'multi injection moldable': 'multi injection molding',
    'multi injection mouldable': 'multi injection molding',
    '2k injection molding': 'multi injection molding',
    '2k injection moulding': 'multi injection molding',
    'double shot molding': 'multi injection molding',
    'double shot moulding': 'multi injection molding',
    'other extrusion': 'other extrusion',
    'otherex': 'other extrusion',
    'power coating / sintering': 'porous sintering',
    'powder coating / sintering': 'porous sintering',
    'porous sintering': 'porous sintering',
    'poweder coating': 'porous sintering',
    'porous sintering': 'porous sintering',
    'sintering': 'porous sintering',
    'profile extrusion': 'profile extrusion',
    'profex': 'profile extrusion',
    'profile': 'profile extrusion',
    'profilextrusion': 'profile extrusion',
    'rotational molding': 'rotational molding',
    'rotomolding': 'rotational molding',
    'rotomoulding': 'rotational molding',
    'rotomoldable': 'rotational molding',
    'rotomouldable': 'rotational molding',
    'rm': 'rotational molding',
    'rotational moulding': 'rotational molding',
    'rotational moldable': 'rotational molding',
    'rotational mouldable': 'rotational molding',
    'selective reinforcement': 'selective reinforcement',
    'sheet extrusion': 'sheet extrusion',
    'sheet': 'sheet extrusion',
    'sheetex': 'sheet extrusion',
    'thermoforming': 'thermoforming',
    'thermo forming': 'thermoforming',
    'thermo': 'thermoforming',
    'transfer molding': 'transfer molding',
    'transfer moulding': 'transfer molding',
    'xfermol': 'transfer molding',
    'transfer moldable': 'transfer molding',
    'transfer mouldable': 'transfer molding',
    
    'calandaring': 'calendering',   
    'gel spinning (fibers)': 'fiber spinning / gel spinning',

}

ignore_processing = [
    'robotic extrusion',
    # 'overmolding', # it is app
]

In [8]:
feature_syn_mapping = {
    'anti-static': 'anti-static',
    'static resistant': 'anti-static',
    'static': 'anti-static',
    'static resistance': 'anti-static',
    'antistatic': 'anti-static',
    'bio-content': 'bio-content',
    'bio-based': 'bio-content',
    'bio': 'bio-content',
    'biobased': 'bio-content',
    'eco-b': 'bio-content',
    'chemical resistant': 'chemical resistant',
    'acid resistant': 'chemical resistant',
    'chemical resistance': 'chemical resistant',
    'acid resistance': 'chemical resistant',
    'chem resistance': 'chemical resistant',
    'chem resistant': 'chemical resistant',
    'anti chemical': 'chemical resistant',
    'anti acid': 'chemical resistant',
    'anti-chemical': 'chemical resistant',
    'anti-acid': 'chemical resistant',
    'contains recycle': 'recycled content',
    'recycled content': 'recycled content',
    'recycle': 'recycled content',
    'recycled': 'recycled content',
    'eco-r': 'recycled content',
    'post consumer recycle': 'recycled content',
    'post-consumer recycle': 'recycled content',
    'pcr': 'recycled content',
    'pir': 'recycled content',
    'post industrial recycle': 'recycled content',
    'post-industrial recycle': 'recycled content',
    'flame retardant': 'flame retardant',
    'flameretardent': 'flame retardant',
    'fr': 'flame retardant',
    'flame retarding agent': 'flame retardant',
    'flamret': 'flame retardant',
    'flamretag': 'flame retardant',
    'flame resistant': 'flame retardant',
    'flame resistance': 'flame retardant',
    'heat stabilized or stable to heat': 'heat stabilized or stable to heat',
    'stable to heat': 'heat stabilized or stable to heat',
    'heat resistance': 'heat stabilized or stable to heat',
    'heat tolerant': 'heat stabilized or stable to heat',
    'high heat performance': 'heat stabilized or stable to heat',
    'high heat': 'heat stabilized or stable to heat',
    'heat stabilized': 'heat stabilized or stable to heat',
    'heat stabilised': 'heat stabilized or stable to heat',
    'heat': 'heat stabilized or stable to heat',
    'heat stable': 'heat stabilized or stable to heat',
    'high heat conditions': 'heat stabilized or stable to heat',
    'resistant to heat': 'heat stabilized or stable to heat',
    'hhr': 'heat stabilized or stable to heat',
    'thermal resistance': 'heat stabilized or stable to heat',
    'thermal resistant': 'heat stabilized or stable to heat',
    'anti heat': 'heat stabilized or stable to heat',
    'anti-heat': 'heat stabilized or stable to heat',
    'high flow': 'high flow',
    'low mv': 'high flow',
    'high flowability': 'high flow',
    'low viscosity': 'high flow',
    'easy flow': 'high flow',
    'thin wall': 'high flow',
    'low wall': 'high flow',
    'thin walled': 'high flow',
    'high gloss': 'high gloss',
    'enhanced gloss': 'high gloss',
    'highgloss': 'high gloss',
    'impact modified': 'high impact or impact modified',
    'high impact or impact modified': 'high impact or impact modified',
    'toughened': 'high impact or impact modified',
    'impact resistance': 'high impact or impact modified',
    'impact resistant': 'high impact or impact modified',
    'anti impact': 'high impact or impact modified',
    'anti-impact': 'high impact or impact modified',
    'impact grade': 'high impact or impact modified',
    'impact modification': 'high impact or impact modified',
    'improved toughness': 'high impact or impact modified',
    'tough': 'high impact or impact modified',
    'high viscosity': 'high viscosity',
    'high mv': 'high viscosity',
    'low flow': 'high viscosity',
    'highvisc': 'high viscosity',
    'hydrolysis resistant': 'hydrolysis resistant',
    'hydrolysis': 'hydrolysis resistant',
    'water resistant': 'hydrolysis resistant',
    'anti hydrolysis': 'hydrolysis resistant',
    'hydrolysis resistance': 'hydrolysis resistant',
    'water resistance': 'hydrolysis resistant',
    'anti-hydrolysis': 'hydrolysis resistant',
    'anti-water': 'hydrolysis resistant',
    'hydrostable': 'hydrolysis resistant',
    # 'hygroscopic': 'hydrolysis resistant',
    'not hygroscopic': 'hydrolysis resistant',
    'improved creep': 'improved creep',
    'creep resistance': 'improved creep',
    'creep resistant': 'improved creep',
    'anti creep': 'improved creep',
    'anti-creep': 'improved creep',
    'improved weld line': 'improved weld line',
    'weld line improved': 'improved weld line',
    'weld line strength': 'improved weld line',
    'increased electrical conductivity': 'increased electrical conductivity',
    # 'conductive': 'increased electrical conductivity',
    'electro conductive': 'increased electrical conductivity',
    'electrical conductivity': 'increased electrical conductivity',
    'increased thermal conductivity': 'increased thermal conductivity',
    'laser direct structurable': 'laser direct structurable',
    'lds': 'laser direct structurable',
    'laser direct structuring': 'laser direct structurable',
    'laser direct structuring (lds)': 'laser direct structurable',
    'laser markable': 'laser markable',
    'lm': 'laser markable',
    'lasermarkable': 'laser markable',
    'laser marking': 'laser markable',
    'lasermark': 'laser markable',
    'laser weldable': 'laser weldable',
    'lt': 'laser weldable',
    'laser transparent': 'laser weldable',
    'lasertr': 'laser weldable',
    'lasertransparent': 'laser weldable',
    'laser weld': 'laser weldable',
    'laser transmission': 'laser weldable',
    'lead-free soldering resistant': 'lead-free soldering resistant',
    'surface mount solderable': 'lead-free soldering resistant',
    'smd solderable': 'lead-free soldering resistant',
    'lead-free reflow solderable': 'lead-free soldering resistant',
    'lead-free solderability': 'lead-free soldering resistant',
    'lead-free soldering resistance': 'lead-free soldering resistant',
    'pb free solder reflow': 'lead-free soldering resistant',
    'anti lead free soldering': 'lead-free soldering resistant',
    'light stabilized or stable to light': 'light stabilized or stable to light',
    'light stabilised or stable to light': 'light stabilized or stable to light',
    'light stabilised': 'light stabilized or stable to light',
    'light stabilized': 'light stabilized or stable to light',
    'stable to light': 'light stabilized or stable to light',
    'light weight': 'light weight',
    'light weighting': 'light weight',
    'replacement for aluminum': 'light weight',
    'metal replacement': 'light weight',
    'low emissions': 'low emissions',
    'low emission': 'low emissions',
    'reduced emission': 'low emissions',
    'low-e': 'low emissions',
    'lowemiss': 'low emissions',
    'emissions': 'low emissions',
    'low formaldehyde': 'low emissions',
    'low halide content': 'low halide content',
    'low warpage': 'low warpage',
    'low warp': 'low warpage',
    'warp resistant': 'low warpage',
    'lowwarp': 'low warpage',
    'high flatness': 'low warpage',
    'warp resistance': 'low warpage',
    'anti warp': 'low warpage',
    'anti-warp': 'low warpage',
    'low wear / low friction': 'low wear / low friction',
    'low wear': 'low wear / low friction',
    'low friction': 'low wear / low friction',
    'tribo': 'low wear / low friction',
    'tribological': 'low wear / low friction',
    'low wear and friction': 'low wear / low friction',
    'tribological modified': 'low wear / low friction',
    'tribology': 'low wear / low friction',
    'good sliding': 'low wear / low friction',
    'lw': 'low wear / low friction',
    'lubricants': 'lubricants',
    'lubricated': 'lubricants',
    'lubric': 'lubricants',
    'medical/healthcare': 'medical/healthcare',
    'medical': 'medical/healthcare',
    'medical grade': 'medical/healthcare',
    'medgrade': 'medical/healthcare',
    'medhc': 'medical/healthcare',
    'medical grades': 'medical/healthcare',
    'healthcare': 'medical/healthcare',
    'medical grade technology': 'medical/healthcare',
    'mt': 'medical/healthcare',
    'med': 'medical/healthcare',
    'non-halogenated/red phosphorous free flame retardant': 'non-halogenated/red phosphorous free flame retardant',
    'halogen free': 'non-halogenated/red phosphorous free flame retardant',
    'haogen free': 'non-halogenated/red phosphorous free flame retardant',
    'nhfr': 'non-halogenated/red phosphorous free flame retardant',
    'hffr': 'non-halogenated/red phosphorous free flame retardant',
    'halogen free flame retardant': 'non-halogenated/red phosphorous free flame retardant',
    'non-halogenated': 'non-halogenated/red phosphorous free flame retardant',
    'non-halogenated fr': 'non-halogenated/red phosphorous free flame retardant',
    'nucleated': 'nucleated',
    'high crystallinity': 'nucleated',
    'highly crystalline': 'nucleated',
    'highest cristalline': 'nucleated',
    'plasticizer': 'plasticizer',
    'plasticiser': 'plasticizer',
    'plasti': 'plasticizer',
    'platable': 'platable',
    'plateable': 'platable',
    'plating grade': 'platable',
    'reduced gloss': 'reduced gloss',
    'low gloss': 'reduced gloss',
    'redugloss': 'reduced gloss',
    'release agent': 'release agent',
    'release': 'release agent',
    'specialty appearance': 'specialty appearance',
    'lx': 'specialty appearance',
    'metalx': 'specialty appearance',
    'appearance': 'specialty appearance',
    'aesthetical': 'specialty appearance',
    'good aesthetics': 'specialty appearance',
    'visual fx': 'specialty appearance',
    'static dissipative': 'static dissipative',
    'esd': 'static dissipative',
    'esd modified': 'static dissipative',
    'elecdiss': 'static dissipative',
    'elec diss': 'static dissipative',
    'electdiss': 'static dissipative',
    'elect diss': 'static dissipative',
    'electrostatic dissipation': 'static dissipative',
    'sustainable': 'sustainable',
    'green': 'sustainable',
    'circular economy': 'sustainable',
    'eco': 'sustainable',
    'environmentally friendly': 'sustainable',
    'sustainability': 'sustainable',
    'u.v. stabilized': 'u.v. stabilized or stable to weather',
    'u.v. stabilised': 'u.v. stabilized or stable to weather',
    'u.v. stabilized or stable to weather': 'u.v. stabilized or stable to weather',
    'u.v. stabilised or stable to weather': 'u.v. stabilized or stable to weather',
    'uv resistant': 'u.v. stabilized or stable to weather',
    'uv': 'u.v. stabilized or stable to weather',
    'anti uv': 'u.v. stabilized or stable to weather',
    'anti-uv': 'u.v. stabilized or stable to weather',
    'uv-stabilized': 'u.v. stabilized or stable to weather',
    'exposure to outdoor sunlight': 'u.v. stabilized or stable to weather',
    'uv resistent': 'u.v. stabilized or stable to weather',
    'uvres': 'u.v. stabilized or stable to weather',
    'uv stabilise': 'u.v. stabilized or stable to weather',
    'uv stability': 'u.v. stabilized or stable to weather',
    'weather resistance': 'u.v. stabilized or stable to weather',
    'weather resistant': 'u.v. stabilized or stable to weather',
    'uv resistance': 'u.v. stabilized or stable to weather',
    'weatherable': 'u.v. stabilized or stable to weather',
    'stable to weather': 'u.v. stabilized or stable to weather',
    'weather': 'u.v. stabilized or stable to weather',
    'ultrasonic weldable': 'ultrasonic weldable',
    
    'flame-reatardant': 'flame retardant',
    'low wear/low friction': 'low wear / low friction',
    'highest christalline': 'nucleated',
    'flatness': 'low warpage',
    'bio-compatability': 'medical/healthcare',
    'non-halogenated/red phosphorus free flame retardant': 'non-halogenated/red phosphorous free flame retardant',
    'high impact': 'high impact or impact modified',
    'laser transparant': 'laser weldable',
    # 'non sustainable': 'sustainable', # it is not sustainable, it is wrong
    'thermal conductivity': 'increased thermal conductivity',
    'weatherability': 'u.v. stabilized or stable to weather',  
    'low moisuture': 'hydrolysis resistant',
    'increased elctrical conductivity': 'increased electrical conductivity',
    'electric conductive': 'increased electrical conductivity',
    'water glycol resistance': 'water glycol resistant',
    'thermally conductive': 'increased thermal conductivity',
    'high fatigue resistance': 'fatigue resistant',
    'plateble': 'platable',
    'modified impact': 'high impact or impact modified',
    'fuel resistance': 'fuel resistant',
    'outdoor protection and weatherability': 'u.v. stabilized or stable to weather',
    'hydro resistance': 'hydrolysis resistant',
    'chemical compatability': 'chemical resistant',
    'low emision': 'low emissions',
    'red phosphorous': 'non-halogenated/red phosphorous free flame retardant',
    'outdoor weathering/uv stabilized': 'u.v. stabilized or stable to weather',
    'oil resistance': 'oil resistant',
    'low warpag': 'low warpage',
    'high welding strength': 'improved weld line',
    'highly toughened': 'high impact or impact modified',
    'rec': 'recycled content',
    'shock resistance': 'thermal shock resistant',
    'resistance to fluorine': 'fluorine resistant',
    'non-hal': 'non-halogenated/red phosphorous free flame retardant',
    'laser welding': 'laser weldable',
    'recyled content': 'recycled content',
    'weld strength': 'improved weld line',
    'special appearance': 'specialty appearance',
    'ultrasonic welding': 'ultrasonic weldable',
    'wear resistance': 'low wear / low friction',
    # 'reflow soldering': 'lead-free soldering resistant',
    'wr': 'low wear / low friction',
    'lubricant': 'lubricants',
    'resistant to acetone': 'chemical resistant',
    'hot oil resistant': 'oil resistant',
    'soldering': 'lead-free soldering resistant',
    'lead-free soldering': 'lead-free soldering resistant',
    'recyclable content': 'recycled content',
    'hydolysis resistant': 'hydrolysis resistant',
    'heat resistant': 'heat stabilized or stable to heat',
    'recycle content': 'recycled content',
    'restistance to water': 'hydrolysis resistant',
    'smt': 'smt process',
    'odor': 'low odor',
    'oil resitance': 'oil resistant',
    'recycled content': 'recycled content',
    'friction': 'low wear / low friction',
    'surface appearance': 'specialty appearance',
    'red phosphor based flame retardant': 'non-halogenated/red phosphorous free flame retardant',
    'weldline strength': 'improved weld line',
    'high weld strength': 'improved weld line',
    'minimal warpage': 'low warpage',
    'wear friction': 'low wear / low friction',
    'recycled contents': 'recycled content',
    'electrostatically': 'static dissipative',
    'wear reisitance': 'low wear / low friction',
    'high weld line strength': 'improved weld line',
    'non halogenated': 'non-halogenated/red phosphorous free flame retardant',
    'resistant to high temperatures': 'heat stabilized or stable to heat',
    'resistant to atf oil': 'oil resistant',
    'lead free soldering': 'lead-free soldering resistant',
    'flame retradant': 'flame retardant',
    # 'hydrolytically stable': 'hydrolysis resistant', # ignored based on stacie's review
    'resistance hydrocarbons': 'fuel resistant',
    'outdoor weathering / uv stabilized': 'u.v. stabilized or stable to weather',
    'emi': 'emi shielding',
    'acetone resistance': 'chemical resistant',
    'resistant to blooming': 'blooming resistant',
    'non-chlorine': 'chlorine resistant',
    'hydrolyses stabilized': 'hydrolysis resistant',
    'thermal shock': 'thermal shock resistant',
    'electrically conductive': 'increased electrical conductivity',
    'transparent': 'laser weldable',
    'nh': 'non-halogenated/red phosphorous free flame retardant',
    'wear resis': 'low wear / low friction',
    'halogene-free': 'non-halogenated/red phosphorous free flame retardant',
    'weld line': 'improved weld line',
                       
    'reflow soldering': 'reflow soldering',
    'abrasion resistance': 'low wear / low friction',
    'weldable': 'weldable',
    'weld': 'weld',
    'resistant': 'resistant',
    'vibration welding': 'ultrasonic weldable',
    'recoverable compliance': 'improved creep',
    'chrome-platable': 'platable',
    'chrome-plated': 'platable',
    'small particle': 'powder',
    'welding': 'welding',
    'temperature resistance': 'heat stabilized or stable to heat',
    'laser': 'laser',
    'conductive': 'conductivity',
    'blooming free': 'blooming resistant',
    'cold temperature performance': 'cold temperature performance',
    'glycol contact': 'glycol contact',
    'silicon friendly': 'silicon friendly',
    'retortable': 'retortable',    
    'optically transparent': 'optically transparent',
    'no blister': 'blister resistant',
    'non-bromine': 'bromine resistant',
    'fungus resistant': 'fungus resistant',
    'whitening free': 'whitening resistant',
    'ductility': 'ductility',
    'thermal shock resistance': 'thermal shock resistant',
    'anti-thermal shock': 'thermal shock resistant',
    'coolant resistant': 'coolant resistant',
    'heat dissipation': 'increased thermal conductivity',
    'hr': 'hr',
    'carbon free': 'carbon capture',
    'carbon neutral': 'carbon capture',
    'tf': 'low wear / low friction',
    'solar resistance': 'u.v. stabilized or stable to weather',
    'solar resistant': 'u.v. stabilized or stable to weather',
}

ignore_feature = [
    'non sustainable',
    'crystallinity',
    'barnacle resistance',
    'medical & pharma - drug delivery devices',
    'low ion elution',
    'heat aging',
    'oxidation',
    'steam sterilization',
    'sardine flavor',
    'insect resistant',    
]

In [9]:
delivery_syn_mapping = {
    'granules': 'granules',
    'pellets': 'pellets',
    'pelets': 'pellets',
    'pellet': 'pellets',
    'powder': 'powder',
    'tape': 'tape',
    'tp': 'tape',
    'cfr': 'tape',
}

In [10]:
polymer_syn_mapping = {
    'abs': 'abs',
    'acrylonitrile-butadiene-styrene': 'abs',
    'lcp': 'lcp',
    'liquid crystal polymer': 'lcp',
    'liquid - crystalline polymer': 'lcp',
    'lcpa': 'lcpa',
    'long chain nylon': 'lcpa',
    'long chain pa': 'lcpa',
    'pa*': 'pa*',
    'polyamide': 'pa*',
    'pa': 'pa*',
    'nylon': 'pa*',
    'pa1010': 'pa1010',
    'polyamide 1010': 'pa1010',
    'nylon 1010': 'pa1010',
    'pa 1010': 'pa1010',
    'polyamide 10/10': 'pa1010',
    'nylon 10/10': 'pa1010',
    'pa10/10': 'pa1010',
    'pa 10/10': 'pa1010',
    'polyamide 10,10': 'pa1010',
    'nylon 10,10': 'pa1010',
    'pa 10,10': 'pa1010',
    'pa10,10': 'pa1010',
    'polyamide 10-10': 'pa1010',
    'nylon 10-10': 'pa1010',
    'pa 10-10': 'pa1010',
    'pa10-10': 'pa1010',
    'pa6': 'pa6',
    'nylon 6': 'pa6',
    'pa 6': 'pa6',
    'polyamide 6': 'pa6',
    'pa610': 'pa610',
    'polyamide 610': 'pa610',
    'nylon 610': 'pa610',
    'pa 610': 'pa610',
    'nylon 6-10': 'pa610',
    'polyamide 6-10': 'pa610',
    'pa 6-10': 'pa610',
    'pa6-10': 'pa610',
    'nylon 6/10': 'pa610',
    'polyamide 6/10': 'pa610',
    'pa 6/10': 'pa610',
    'pa6/10': 'pa610',
    'nylon 6,10': 'pa610',
    'polyamide 6,10': 'pa610',
    'pa 6,10': 'pa610',
    'pa6,10': 'pa610',
    'pa612': 'pa612',
    'polyamide 612': 'pa612',
    'nylon 612': 'pa612',
    'pa 612': 'pa612',
    'nylon 6-12': 'pa612',
    'polyamide 6-12': 'pa612',
    'pa 6-12': 'pa612',
    'pa6-12': 'pa612',
    'nylon 6/12': 'pa612',
    'polyamide 6/12': 'pa612',
    'pa 6/12': 'pa612',
    'pa6/12': 'pa612',
    'nylon 6,12': 'pa612',
    'polyamide 6,12': 'pa612',
    'pa 6,12': 'pa612',
    'pa6,12': 'pa612',
    'pa66': 'pa66',
    'polyamide 66': 'pa66',
    'nylon 66': 'pa66',
    'pa 66': 'pa66',
    'nylon 6-6': 'pa66',
    'polyamide 6-6': 'pa66',
    'pa 6-6': 'pa66',
    'pa6-6': 'pa66',
    'nylon 6/6': 'pa66',
    'polyamide 6/6': 'pa66',
    'pa 6/6': 'pa66',
    'pa6/6': 'pa66',
    'nylon 6,6': 'pa66',
    'polyamide 6,6': 'pa66',
    'pa 6,6': 'pa66',
    'pa6,6': 'pa66',
    'pa66/6t': 'pa66/6t',
    'pa666': 'pa666',
    'polyamide 666': 'pa666',
    'nylon 666': 'pa666',
    'pa 666': 'pa666',
    'nylon 66-6': 'pa666',
    'polyamide 66-6': 'pa666',
    'pa 66-6': 'pa666',
    'pa66-6': 'pa666',
    'nylon 66/6': 'pa666',
    'polyamide 66/6': 'pa666',
    'pa 66/6': 'pa666',
    'pa66/6': 'pa666',
    'nylon 66,6': 'pa666',
    'polyamide 66,6': 'pa666',
    'pa 66,6': 'pa666',
    'pa66,6': 'pa666',
    'pa6t/66': 'pa6t/66',
    'pa6t/6i': 'pa6t/6i',
    '6t6i': 'pa6t/6i',
    'pa6t/xt': 'pa6t/xt',
    '6txt': 'pa6t/xt',
    'pbt': 'pbt',
    'polybutylene terephthalate': 'pbt',
    'polyester': 'pxt',
    'pc': 'pc',
    'polycarbonate': 'pc',
    'pct': 'pct',
    'polycyclohexylenedimethylene terephthalate': 'pct',
    'pe-hd': 'pe-hd',
    'hdpe': 'pe-hd',
    'high density polyethylene': 'pe-hd',
    'pehd': 'pe-hd',
    'hd-pe': 'pe-hd',
    'pe-hmw': 'pe-hmw',
    'hmwpe': 'pe-hmw',
    'high molecular weight polyethylene': 'pe-hmw',
    'hmw-pe': 'pe-hmw',
    'pe-uhmw': 'pe-uhmw',
    'uhmwpe': 'pe-uhmw',
    'ultra high molecular weight polyethylene': 'pe-uhmw',
    'uhmw-pe': 'pe-uhmw',
    'uhmw': 'pe-uhmw',
    'umpe': 'pe-uhmw',
    'pet': 'pet',
    'polyethylene terephthalate': 'pet',
    'pom': 'pom',
    'polyoxymethylene (acetal)': 'pom',
    'polyoxymethylene': 'pom',
    'acetal copolymer': 'pom',
    'polyformaldehyde': 'pom',
    'copolymer': 'pom',
    'acetal': 'pom',
    'polyacetal': 'pom',
    'pp': 'pp',
    'polypropylene': 'pp',
    'ppa': 'ppa',
    'polyphthalamide': 'ppa',
    'semi aromatic polyamide': 'ppa',
    'semi aromatic pa': 'ppa',
    'pps': 'pps',
    'polyphenylene sulfide': 'pps',
    'polyphenylene sulphide': 'pps',
    'tpc': 'tpc',
    'copolyester thermoplastic elastomer': 'tpc',
    'thermoplastic copolyester': 'tpc',
    'copolyester elastomer': 'tpc',
    'tpc-et': 'tpc',
    'cope': 'tpc',
    'tpe': 'tpe',
    'thermoplastic elastomer': 'tpe',
    'tpu': 'tpu',
    'thermoplastic polyurethane': 'tpu',
    'tpv': 'tpv',
    'thermoplastic vulcanizate': 'tpv',
    'thermoplastic vulcanizates': 'tpv',
    'thermoplastic rubber vulcanisate': 'tpv',
                       
    'polyproplyene': 'pp',
    'high molecular weight pe': 'pe-hmw', 
    'pehmw': 'pe-hmw', 
    'semi aromatic nylon': 'ppa',
    'polyphenylene sulfide ( pps )': 'pps',
    'polyamid': 'pa*',
    'polyoxymethylene ( pom )': 'pom',
    # 'elastomnwer': 'elastomer',
    'elastomnwer': 'tpe', # from polymer family
    'polyamide 55': 'pa55',
    'polyphenylene sulfide (pps)': 'pps',
    'thermoplastic vulcanizate ( tpv )': 'tpv',
    'polypropylene ( pp )': 'pp',
    'semi - aromatic nylon': 'ppa',
    'tpvs': 'tpv',
    'polyoxymethylene (pom)': 'pom',
    'polyamid 6': 'pa6',
    'polypropylene (pp)': 'pp',
    'liquid crystal polymers': 'lcp',
    'liquid crystal polymer (lcp)': 'lcp',
    'semi-aromatic pa': 'ppa',
    'polyphenyle sulfide': 'pps',
    'polyphenylens sulfide': 'pps',
    'pa6t xt': 'pa6t/xt',
    'acetyl copolymer': 'pom',
    'liquid crystalline polyester': 'lcp',
    'polypropolyene': 'pp',
    'polypropylyne': 'pp',
    'liquid crystal': 'lcp',
    'ultra - high molecular weight polyethylene': 'pe-uhmw',
    'ultra-high molecular weight polyethylene': 'pe-uhmw',
    'pa nylon': 'pa',
    'polyethylene': 'pe',
    'polyphenylene sulfide pps': 'pps',
    'polyamide 6 ( pa6 )': 'pa6',
    'semi - aromatic pa': 'ppa',
    'polypthalamide': 'ppa',
    'polipropilene': 'pp',
    'nylon66': 'pa66',
    'polycarboante': 'pc',
    'polyphthalamides': 'ppa',
    'semi-aromatic nylon': 'ppa',
    'pa6.12': 'pa612',
    'polyphenylene': 'pps',
    'thermoplastic vulcanate': 'tpv',
    'thermoplastic copolymer': 'tpc',
    'pa6/66': 'pa666',
    'hmw': 'pe-hmw',
    
    'liquid crystal polymer ( lcp )': 'lcp', 
    'ldpe': 'pe-ld',
    'polyketone': 'pk', 
    'ethylene vinyl acetate': 'evac',
    'semi - crystalline polyamide': 'pa',
    'semi-crystalline polyamide': 'pa',
    'eva': 'evac',
    'acrylic elastomer': 'aem',
    'ethylene acrylic elastomer': 'aem',
    'homopolymer': 'pom',
    'thermoplastics': 'thermoplastic',
    'sebs': 'sebs',
    'ptfe': 'ptfe',
    'sbs': 'sbs',
    'peek': 'peek',
    
    'polyphthalamide': 'ppa',
    'high performance nylon': 'ppa',
    'hpn': 'ppa',
    'high performance polyamide': 'ppa',
    'long chain polyamide' : 'lcpa',
    'elastomer': 'tpe',
}

ignore_polymers = [
    'asa',
    'pvdf',
    'epdm',
    'pvc',
    
]

In [11]:
brand_syn_mapping = {
    'celanex': 'celanex',
    'cx': 'celanex',
    'celanyl': 'celanyl',
    'cnl': 'celanyl',
    'celcon': 'celcon',
    'cn': 'celcon',
    'celstran': 'celstran',
    'cs': 'celstran',
    'lft': 'celstran',
    'lfrt': 'celstran',
    'cs lft': 'celstran',
    'celstran lft': 'celstran',
    'pultrusion': 'celstran',
    'lftr': 'celstran',
    'coolpoly': 'coolpoly',
    'cp': 'coolpoly',
    'crastin': 'crastin',
    'cra': 'crastin',
    'ecomid': 'ecomid',
    'fortron': 'fortron',
    'fo': 'fortron',
    'frianyl': 'frianyl',
    'gur': 'gur',
    'hostaform': 'hostaform',
    'hf': 'hostaform',
    'hytrel': 'hytrel',
    'hyt': 'hytrel',
    # 'htr': 'hytrel', # grade
    'impet': 'impet',
    'im': 'impet',
    'kepital': 'kepital',
    'kep': 'kepital',
    'rynite': 'rynite',
    'santoprene': 'santoprene',
    'stp': 'santoprene',
    'thermx': 'thermx',
    'tx': 'thermx',
    'vandar': 'vandar',
    'va': 'vandar',
    'vectra': 'vectra',
    've': 'vectra',
    'zenite': 'zenite',
    'ze': 'zenite',
    'zytel': 'zytel',
    'zyt': 'zytel',
    
    'po': 'polifor',          
    'selar': 'selar',
    'elvamide': 'elvamide',
    'polifor': 'polifor',
    'amcel': 'amcel',
    'fp': 'forprene',
    'pibiflex': 'pibiflex',
    'eco': 'ecomid',
    'ff': 'forflex',
    'lp': 'laprene',
    'geolast': 'geolast',
    'tc': 'tecnoprene',
    'ateva': 'ateva',
    'vamac': 'vamac',
    'tecnoprene': 'tecnoprene',

}

ignore_brand = [
    'pb'
]

In [12]:
auto_syns_mapping = {
    'daimler': 'mercedes-benz',
    'daimler-benz': 'mercedes-benz',
    'daimler mercedes': 'mercedes-benz',
    'daimler mercedes-benz': 'mercedes-benz',
    'daimler mercedes-benz group': 'mercedes-benz',
    'mercedes-benz': 'mercedes-benz',
    'vw group': 'vw group',
    'vw': 'vw group',
    'bentley': 'vw group',
}

In [13]:
prop_syn_mapping = {
    'electric strength': 'electric strength iec 60243-1 (kv/mm)',
    'specific heat capacity of melt': 'specific heat capacity of melt iso 22007-4 (j/(kg k))',
    'charpy impact strength': 'charpy impact strength',
    'humidity absorption': 'humidity absorption sim. to iso 62 2mm (%)',
    'izod notched impact strength': 'izod notched impact strength',
    'surface resistivity': 'surface resistivity iec 62631-3-2 (ohm)',
    'ball indentation hardness': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'thermal conductivity': 'thermal conductivity',
    'tape thickness': 'tape thickness iso 16012 (mm)',
    'flexural strength': 'flexural strength',
    'charpy notched impact strength': 'charpy notched impact strength',
    'fiber areal weight': 'fiber areal weight - (g/m²)',
    'stress at 10% elongation': 'stress at 10% elongation iso 527-1/-2 or iso 37 (mpa)',
    'oxygen index': 'oxygen index iso 4589-1/-2 (%)',
    'tensile creep modulus': 'tensile creep modulus',
    'coefficient of linear thermal expansion (clte)': 'coefficient of linear thermal expansion (clte)',
    'tensile strain at break': 'tensile strain at break',
    'puncture energy': 'puncture energy',
    'volume resistivity': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'compression set': 'compression set',
    'dissipation factor': 'dissipation factor',
    'average molecular weight': "average molecular weight margolies' equation (g/mol)",
    'izod impact strength': 'izod impact strength',
    'elongational stress f': 'elongational stress f iso 21304-2 150/10 (mpa)',
    'continuous service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'tensile stress at break': 'stress at break',
    'molding shrinkage': 'molding shrinkage',
    'tensile strain at yield': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'glass transition temperature': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
    'tear strength': 'tear strength iso 34-1 normal (kn/m)',
    'effective thermal diffusivity': 'effective thermal diffusivity',
    'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
    'tensile strain at failure': 'tensile strain at failure astm d 3039 m tape 0° (%)',
    'shore d hardness': 'shore d hardness iso 48-4 / iso 868 15s',
    'temperature of deflection under load': 'temperature of deflection under load',
    'intrinsic viscosity': 'intrinsic viscosity iso 307, 1628',
    'compressive stress at 1% strain': 'compressive stress at 1% strain iso 604 (mpa)',
    'flexural strain at failure': 'flexural strain at failure',
    'stress at break': 'stress at break',
    'average particle size': 'average particle size laser scattering d50 (µm)',
    'melt mass-flow rate': 'melt mass-flow rate iso 1133 (g/10min)',
    'wear by sandslurry method (based on gur 4120=100)': 'wear by sandslurry method (based on gur 4120=100)',
    'viscosity number': 'viscosity number iso 307, 1628 (cm³/g)',
    'fiber volume content': 'fiber volume content iso 11667 (%)',
    'flexural modulus': 'flexural modulus',
    'tensile modulus': 'tensile modulus',
    'hardness, rockwell': 'hardness, rockwell',
    'compressive strength': 'compressive strength iso 604 (mpa)',
    'tape width': 'tape width iso 16012 (mm)',
    'tensile stress at 50% strain': 'tensile stress at 50% strain iso 527-1/-2 (mpa)',
    'nominal strain at break': 'nominal strain at break iso 527-1/-2 (%)',
    'melting temperature': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
    'compressive modulus': 'compressive modulus iso 604 (mpa)',
    'relative permittivity': 'relative permittivity',
    'water absorption': 'water absorption sim. to iso 62 2mm (%)',
    'ball pressure test': 'ball pressure test iec 60695-10-2 (°c)',
    'bulk density': 'bulk density iso 60 (kg/m³)',
    'density': 'density iso 1183 (kg/m³)',
    'charpy double notched impact strength': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'flexural stress at 3.5%': 'flexural stress at 3.5% iso 178 (mpa)',
    'puncture - maximum force': 'puncture - maximum force iso 6603-2 23°c (n)',
    'tensile notched impact strength': 'tensile notched impact strength iso 8256/1 23°c (kj/m²)',
    'melt volume-flow rate': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'tensile stress at 100% strain': 'tensile stress at 100% strain iso 527-1/-2 (mpa)',
    'tape areal weight': 'tape areal weight - (g/m²)',
    'tensile stress at yield': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'vicat softening temperature': 'vicat softening temperature',
    "poisson's ratio": "poisson's ratio",
    'elongation at break': 'tensile strain at break',
    'tensile strength': 'tensile strength',
    'tape tens mod': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tensile mod tape': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tape tensile modulus': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tensile modulus tape': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tensile tape': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tape flex mod': 'flexural modulus astm d 790 tape 0° (mpa)',
    'flex mod tape': 'flexural modulus astm d 790 tape 0° (mpa)',
    'tape flexural modulus': 'flexural modulus astm d 790 tape 0° (mpa)',
    'flexural modulus tape': 'flexural modulus astm d 790 tape 0° (mpa)',
    'tape flex strength': 'flexural strength astm d 790 tape 0° (mpa)',
    'flex strength tape': 'flexural strength astm d 790 tape 0° (mpa)',
    'tape flexural strength': 'flexural strength astm d 790 tape 0° (mpa)',
    'flexural strength tape': 'flexural strength astm d 790 tape 0° (mpa)',
    "average molecular weight margolies' equation (g/mol)": "average molecular weight margolies' equation (g/mol)",
    'average molar mass': "average molecular weight margolies' equation (g/mol)",
    'average molecular mass': "average molecular weight margolies' equation (g/mol)",
    'avermw': "average molecular weight margolies' equation (g/mol)",
    'molecular weight': "average molecular weight margolies' equation (g/mol)",
    'mw': "average molecular weight margolies' equation (g/mol)",
    'average particle size laser scattering d50 (µm)': 'average particle size laser scattering d50 (µm)',
    'avg. particle size': 'average particle size laser scattering d50 (µm)',
    'avpartsize': 'average particle size laser scattering d50 (µm)',
    'ball indentation hardness iso 2039-1 h 358/30 (mpa)': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'ball indention hardness': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'ball pressure test iec 60695-10-2 (°c)': 'ball pressure test iec 60695-10-2 (°c)',
    'bulk density iso 60 (kg/m³)': 'bulk density iso 60 (kg/m³)',
    'apparent (bulk) density': 'bulk density iso 60 (kg/m³)',
    'apparent density': 'bulk density iso 60 (kg/m³)',
    'bulk specific gravity': 'bulk density iso 60 (kg/m³)',
    'bulkdens': 'bulk density iso 60 (kg/m³)',
    'charpy double notched impact strength iso 21304-2 23°c (kj/m²)': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'charpy impact strength (double notched)': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'charpy impact strength (double notch)': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notch charpy impact strength': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notched charpy impact strength': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notch charpy': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notched charpy': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notch impact': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'double notched impact': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'charpy double notch': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'charpy double notched': 'charpy double notched impact strength iso 21304-2 23°c (kj/m²)',
    'charpy': 'charpy',
    'charpy impact': 'charpy impact strength',
    'impact': 'impact strength',
    'impact charpy': 'charpy impact strength',
    'impact unnotched': 'charpy impact strength',
    'impact unnotch': 'charpy impact strength',
    'unnotched impact': 'charpy impact strength',
    'unnotch impact': 'charpy impact strength',
    'charpy unnotched': 'charpy impact strength',
    'charpy unnotch': 'charpy impact strength',
    'unnotched charpy': 'charpy impact strength',
    'unnotch charpy': 'charpy impact strength',
    'charpy unnotched impact': 'charpy impact strength',
    'charpy unnotch impact': 'charpy impact strength',
    'unnotched charpy impact': 'charpy impact strength',
    'unnotch charpy impact': 'charpy impact strength',
    'charpy impact strength unnotched': 'charpy impact strength',
    'charpy impact strength unnotch': 'charpy impact strength',
    'charpy impact strength (unnotched)': 'charpy impact strength',
    'charpy impact strength (unnotch)': 'charpy impact strength',
    'charpy impact unnotched': 'charpy impact strength',
    'charpy impact unnotch': 'charpy impact strength',
    'charpy impact (unnotched)': 'charpy impact strength',
    'impact notched': 'charpy notched impact strength',
    'impact notch': 'charpy notched impact strength',
    'notched impact': 'notched impact',
    'notch impact': 'charpy notched impact strength',
    'charpy notched': 'charpy notched impact strength',
    'charpy notch': 'charpy notched impact strength',
    'notched charpy': 'charpy notched impact strength',
    'notch charpy': 'charpy notched impact strength',
    'charpy notched impact': 'charpy notched impact strength',
    'charpy notch impact': 'charpy notched impact strength',
    'notched charpy impact': 'charpy notched impact strength',
    'notch charpy impact': 'charpy notched impact strength',
    'charpy impact strength notched': 'charpy notched impact strength',
    'charpy impact strength notch': 'charpy notched impact strength',
    'charpy impact strength (notched)': 'charpy notched impact strength',
    'charpy impact strength (notch)': 'charpy notched impact strength',
    'charpy impact notched': 'charpy notched impact strength',
    'charpy impact notch': 'charpy notched impact strength',
    'charpy impact (notched)': 'charpy notched impact strength',
    'charpy impact (notch)': 'charpy notched impact strength',
    'impact strength notched': 'charpy notched impact strength',
    'impact strength notch': 'charpy notched impact strength',
    'impa': 'impact strength',
    'coefficient of linear thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'clte': 'coefficient of linear thermal expansion (clte)',
    'linear expansion factor': 'coefficient of linear thermal expansion (clte)',
    'thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'linear thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'coef. of lin. therm expansion': 'coefficient of linear thermal expansion (clte)',
    'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'linear expansion factor (normal)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'normal thermal expansion': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'normal linear thermal expansion': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'coef. of lin. therm expansion normal': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'clte (transverse direction)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 normal (e-6/k)',
    'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'linear expansion factor (parallel)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'clte - parallel': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'parallel thermal expansion': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'parallel linear thermal expansion': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'coef. of lin. therm expansion parallel': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'clte (flow direction)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'compression set iso 815 23°c (%)': 'compression set iso 815 23°c (%)',
    'compression set at 23c': 'compression set iso 815 23°c (%)',
    'compression set iso 815 23°c, 24h (%)': 'compression set iso 815 23°c, 24h (%)',
    'compression set at 23c, 24h': 'compression set iso 815 23°c, 24h (%)',
    'compressive modulus iso 604 (mpa)': 'compressive modulus iso 604 (mpa)',
    'compressive module': 'compressive modulus iso 604 (mpa)',
    'comp mod': 'compressive modulus iso 604 (mpa)',
    'compressive strength iso 604 (mpa)': 'compressive strength iso 604 (mpa)',
    'compressive stress': 'compressive strength iso 604 (mpa)',
    'compressive stress at 1% strain iso 604 (mpa)': 'compressive stress at 1% strain iso 604 (mpa)',
    'compressive strength at 1% strain': 'compressive stress at 1% strain iso 604 (mpa)',
    'compressive stress 1% deformation': 'compressive stress at 1% strain iso 604 (mpa)',
    'compressive strength at 1% deformation': 'compressive stress at 1% strain iso 604 (mpa)',
    'continuous service temperature iec 60216-1 (°c)': 'continuous service temperature iec 60216-1 (°c)',
    'temperature': 'temperature',
    'continuous use temperature': 'continuous service temperature iec 60216-1 (°c)',
    'continuous allowable service temperature in air': 'continuous service temperature iec 60216-1 (°c)',
    'max service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'min service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'service temperature long term': 'continuous service temperature iec 60216-1 (°c)',
    'service temperature short term': 'continuous service temperature iec 60216-1 (°c)',
    'cut': 'continuous service temperature iec 60216-1 (°c)',
    'temperature resistance': 'continuous service temperature iec 60216-1 (°c)',
    'minimum operating temperature': 'continuous service temperature iec 60216-1 (°c)',
    'continuous utilization temperature': 'continuous service temperature iec 60216-1 (°c)',
    'density iso 1183 (kg/m³)': 'density iso 1183 (kg/m³)',
    'dense': 'density iso 1183 (kg/m³)',
    'heavy': 'density iso 1183 (kg/m³)',
    'heavy weight': 'density iso 1183 (kg/m³)',
    'tangent delta': 'dissipation factor',
    'tan delta': 'dissipation factor',
    'df': 'dissipation factor',
    'dielectric dissipation factor': 'dissipation factor',
    'dielectric dissipation': 'dissipation factor',
    'loss angle': 'dissipation factor',
    'loss tangent': 'dissipation factor',
    'dissipation factor iec 62631-2-1 100hz (e-4)': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dissipation factor 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'tangent delta at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'tangent delta, 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'tan delta 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'tan delta at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'df 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    '100hz df': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dielectric dissipation factor at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dielectric dissipation factor, 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dielectric dissipation 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dielectric dissipation at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'loss angle at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'loss angle 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'loss tangent 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'loss tangent at 100hz': 'dissipation factor iec 62631-2-1 100hz (e-4)',
    'dissipation factor iec 62631-2-1 1mhz (e-4)': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'dissipation factor 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'tangent delta at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'tangent delta, 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'tan delta 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'tan delta at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'df 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    '1mhz df': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'dielectric dissipation factor at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'dielectric dissipation factor, 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'dielectric dissipation 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'dielectric dissipation at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'loss angle at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'loss angle 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'loss tangent 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'loss tangent at 1mhz': 'dissipation factor iec 62631-2-1 1mhz (e-4)',
    'effective thermal diffusivity iso 22007-4 crossflow (m²/s)': 'effective thermal diffusivity iso 22007-4 crossflow (m²/s)',
    'effective thermal diffusivity, crossflow direction': 'effective thermal diffusivity iso 22007-4 crossflow (m²/s)',
    'cross plane thermal diffusivity': 'effective thermal diffusivity iso 22007-4 crossflow (m²/s)',
    'effective thermal diffusivity iso 22007-4 flow (m²/s)': 'effective thermal diffusivity iso 22007-4 flow (m²/s)',
    'effective thermal diffusity, flow direction': 'effective thermal diffusivity iso 22007-4 flow (m²/s)',
    'in plane thermal diffusivity': 'effective thermal diffusivity iso 22007-4 flow (m²/s)',
    'effective thermal diffusivity iso 22007-4 through plane (m²/s)': 'effective thermal diffusivity iso 22007-4 through plane (m²/s)',
    'effective thermal diffusivity, through plane direction': 'effective thermal diffusivity iso 22007-4 through plane (m²/s)',
    'through plane thermal diffusivity': 'effective thermal diffusivity iso 22007-4 through plane (m²/s)',
    'electric strength iec 60243-1 (kv/mm)': 'electric strength iec 60243-1 (kv/mm)',
    'ac electristrength': 'electric strength iec 60243-1 (kv/mm)',
    'ac dielectric strength': 'electric strength iec 60243-1 (kv/mm)',
    'ac': 'electric strength iec 60243-1 (kv/mm)',
    'elongation at break iso 527-1/-2 or iso 37 perpendicular (%)': 'elongation at break iso 527-1/-2 or iso 37 perpendicular (%)',
    'elongation at break, perpendicular': 'elongation at break iso 527-1/-2 or iso 37 perpendicular (%)',
    'perpendicular elongation at break': 'elongation at break iso 527-1/-2 or iso 37 perpendicular (%)',
    'perpendicular break strain': 'elongation at break iso 527-1/-2 or iso 37 perpendicular (%)',
    'elongational stress f iso 21304-2 150/10 (mpa)': 'elongational stress f iso 21304-2 150/10 (mpa)',
    'fiber areal weight - (g/m²)': 'fiber areal weight - (g/m²)',
    'fiber volume content iso 11667 (%)': 'fiber volume content iso 11667 (%)',
    'flex modulus': 'flexural modulus',
    'flex mod': 'flexural modulus',
    'flexural modulus of elasticity': 'flexural modulus',
    'flexural performance': 'flexural modulus',
    'rigid': 'flexural modulus',
    'rigidity': 'flexural modulus',
    'bend strength': 'flexural modulus',
    'flexibility': 'flexural modulus',
    'flexible': 'flexural modulus',
    'flex': 'flexural modulus',
    'flexural modulus astm d 790 tape 0° (mpa)': 'flexural modulus astm d 790 tape 0° (mpa)',
    'tape flex modulus': 'flexural modulus astm d 790 tape 0° (mpa)',
    'flexural elongation at break': 'flexural strain at failure',
    'failure flex strain': 'flexural strain at failure',
    'flexural strain at failure astm d 790 tape 0° (%)': 'flexural strain at failure astm d 790 tape 0° (%)',
    'flexural strain at failure, tape': 'flexural strain at failure astm d 790 tape 0° (%)',
    'tape flex strain failure': 'flexural strain at failure astm d 790 tape 0° (%)',
    'flex str': 'flexural strength',
    'flex stress': 'flexural strength',
    'flexural strength astm d 790 tape 0° (mpa)': 'flexural strength astm d 790 tape 0° (mpa)',
    'tape flex str': 'flexural strength astm d 790 tape 0° (mpa)',
    'tape flex stress': 'flexural strength astm d 790 tape 0° (mpa)',
    'flexural stress at 3.5% iso 178 (mpa)': 'flexural stress at 3.5% iso 178 (mpa)',
    'flex stress at 3.5% elongation': 'flexural stress at 3.5% iso 178 (mpa)',
    'flex str at 3.5% strain': 'flexural stress at 3.5% iso 178 (mpa)',
    'glass transition temperature iso 11357-1/-3 10°c/min (°c)': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
    'glass transition': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
    'tg': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
    'rockwell hardness': 'hardness, rockwell',
    'hardness, rockwell iso 2039-2 m-scale': 'hardness, rockwell iso 2039-2 m-scale',
    'hardness, rockwell m-scale': 'hardness, rockwell iso 2039-2 m-scale',
    'rockwell hardness m-scale': 'hardness, rockwell iso 2039-2 m-scale',
    'rockwell, m': 'hardness, rockwell iso 2039-2 m-scale',
    'hardness rockwell m': 'hardness, rockwell iso 2039-2 m-scale',
    'hardness, rockwell iso 2039-2 r-scale': 'hardness, rockwell iso 2039-2 r-scale',
    'hardness, rockwell r-scale': 'hardness, rockwell iso 2039-2 r-scale',
    'rockwell hardness r-scale': 'hardness, rockwell iso 2039-2 r-scale',
    'rockwell, r': 'hardness, rockwell iso 2039-2 r-scale',
    'hardness rockwell r': 'hardness, rockwell iso 2039-2 r-scale',
    'humidity absorption sim. to iso 62 2mm (%)': 'humidity absorption sim. to iso 62 2mm (%)',
    'humid': 'humidity absorption sim. to iso 62 2mm (%)',
    'humidity': 'humidity absorption sim. to iso 62 2mm (%)',
    'moisture absorption': 'humidity absorption sim. to iso 62 2mm (%)',
    'moisture absorption equilibrium 23°c/50% r.h.': 'humidity absorption sim. to iso 62 2mm (%)',
    'intrinsic viscosity iso 307, 1628': 'intrinsic viscosity iso 307, 1628',
    'η': 'intrinsic viscosity iso 307, 1628',
    'intrvisc': 'intrinsic viscosity iso 307, 1628',
    'viscosity': 'viscosity',
    'izod': 'izod',
    'izod impact': 'izod impact strength',
    'impact izod': 'izod impact strength',
    'izod unnotched': 'izod impact strength',
    'izod unnotch': 'izod impact strength',
    'unnotched izod': 'izod impact strength',
    'unnotch izod': 'izod impact strength',
    'izod unnotched impact': 'izod impact strength',
    'izod unnotch impact': 'izod impact strength',
    'unnotched izod impact': 'izod impact strength',
    'unnotch izod impact': 'izod impact strength',
    'izod impact strength unnotched': 'izod impact strength',
    'izod impact strength unnotch': 'izod impact strength',
    'izod impact strength (unnotched)': 'izod impact strength',
    'izod impact strength (unnotch)': 'izod impact strength',
    'izod impact unnotched': 'izod impact strength',
    'izod impact unnotch': 'izod impact strength',
    'izod impact (unnotched)': 'izod impact strength',
    'izod impact (unnotch)': 'izod impact strength',
    'izod notched': 'izod notched impact strength',
    'izod notch': 'izod notched impact strength',
    'notched izod': 'izod notched impact strength',
    'notch izod': 'izod notched impact strength',
    'izod notched impact': 'izod notched impact strength',
    'izod notch impact': 'izod notched impact strength',
    'notched izod impact': 'izod notched impact strength',
    'notch izod impact': 'izod notched impact strength',
    'izod impact strength notched': 'izod notched impact strength',
    'izod impact strength notch': 'izod notched impact strength',
    'izod impact strength (notched)': 'izod notched impact strength',
    'izod impact strength (notch)': 'izod notched impact strength',
    'izod impact notched': 'izod notched impact strength',
    'izod impact notch': 'izod notched impact strength',
    'izod impact (notched)': 'izod notched impact strength',
    'izod impact (notch)': 'izod notched impact strength',
    'melt mass-flow rate iso 1133 (g/10min)': 'melt mass-flow rate iso 1133 (g/10min)',
    'melt mass flow rate': 'melt mass-flow rate iso 1133 (g/10min)',
    'melt flow rate': 'melt mass-flow rate iso 1133 (g/10min)',
    'mfr': 'melt mass-flow rate iso 1133 (g/10min)',
    'flow rate': 'melt mass-flow rate iso 1133 (g/10min)',
    'melt index': 'melt mass-flow rate iso 1133 (g/10min)',
    'melt flow index': 'melt mass-flow rate iso 1133 (g/10min)',
    'mfi': 'melt mass-flow rate iso 1133 (g/10min)',
    'flow': 'melt mass-flow rate iso 1133 (g/10min)',
    'melt volume-flow rate iso 1133 (cm³/10min)': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'melt volume flow rate': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'melt volume rate': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'mvr': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'melt volume': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'mv': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'melting temperature iso 11357-1/-3 10°c/min (°c)': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'melt temp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'dsc melting temp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'mp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'moulding shrinkage': 'molding shrinkage',
    'mold shrinkage': 'molding shrinkage',
    'mold-shrinkage': 'molding shrinkage',
    'mould shrinkage': 'molding shrinkage',
    'dimensional, precise': 'molding shrinkage',
    'dimensional precision': 'molding shrinkage',
    'mold shrinkage (md/td)': 'molding shrinkage',
    'molding shrinkage iso 294-4, 2577 normal (%)': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'molding shrinkage, normal': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mold shrinkage (x-flow)': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mold shrinkage normal': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mold shrinkage(transverse direction)': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mold-shrinkage, normal': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mould shrinkage normal': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mould-shrinkage': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'mould-shrinkage, normal': 'molding shrinkage iso 294-4, 2577 normal (%)',
    'molding shrinkage iso 294-4, 2577 parallel (%)': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'molding shrinkage, parallel': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold shrinkage (flow)': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold shrinkage flow': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold shrinkage machine direction': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold shrinkage md': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold shrinkage parallel': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mold-shrinkage, parallel': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mould shrinkage parallel': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'mould-shrinkage, parallel': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'oxygen index iso 4589-1/-2 (%)': 'oxygen index iso 4589-1/-2 (%)',
    'loi': 'oxygen index iso 4589-1/-2 (%)',
    'limiting oxygen': 'oxygen index iso 4589-1/-2 (%)',
    'poissons ratio': "poisson's ratio",
    'puncture - maximum force iso 6603-2 23°c (n)': 'puncture - maximum force iso 6603-2 23°c (n)',
    'puncture maximum force': 'puncture - maximum force iso 6603-2 23°c (n)',
    'maximum puncture force': 'puncture - maximum force iso 6603-2 23°c (n)',
    'dk': 'relative permittivity',
    'dielectric factor': 'relative permittivity',
    'dielectric loss': 'relative permittivity',
    'relative permittivity iec 62631-2-1 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'relative permittivity at 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dk at 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dielectric factor at 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dielectric loss at 100hz': 'relative permittivity iec 62631-2-1 100hz',
    '100hz dk': 'relative permittivity iec 62631-2-1 100hz',
    'relative permittivity 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dk 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dielectric factor 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'dielectric loss 100hz': 'relative permittivity iec 62631-2-1 100hz',
    'relative permittivity iec 62631-2-1 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'relative permittivity at 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dk at 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dielectric factor at 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dielectric loss at 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    '1mhz dk': 'relative permittivity iec 62631-2-1 1mhz',
    'relative permittivity 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dk 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dielectric factor 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'dielectric loss 1mhz': 'relative permittivity iec 62631-2-1 1mhz',
    'relative permittivity iec 62631-2-1 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'relative permittivity at 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dk at 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dielectric factor at 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dielectric loss at 60hz': 'relative permittivity iec 62631-2-1 60hz',
    '60hz dk': 'relative permittivity iec 62631-2-1 60hz',
    'relative permittivity 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dk 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dielectric factor 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'dielectric loss 60hz': 'relative permittivity iec 62631-2-1 60hz',
    'shore a hardness iso 48-4 / iso 868 15s': 'shore a hardness iso 48-4 / iso 868 15s',
    'shore a': 'shore a hardness iso 48-4 / iso 868 15s',
    'sh a': 'shore a hardness iso 48-4 / iso 868 15s',
    'durometer hardness (a)': 'shore a hardness iso 48-4 / iso 868 15s',
    'durometer hardness (shore a)': 'shore a hardness iso 48-4 / iso 868 15s',
    'a': 'shore a hardness iso 48-4 / iso 868 15s',
    'soft': 'shore a hardness iso 48-4 / iso 868 15s',
    'shore d hardness iso 48-4 / iso 868 15s': 'shore d hardness iso 48-4 / iso 868 15s',
    'shore d': 'shore d hardness iso 48-4 / iso 868 15s',
    'sh d': 'shore d hardness iso 48-4 / iso 868 15s',
    'durometer hardness (d)': 'shore d hardness iso 48-4 / iso 868 15s',
    'durometer hardness (shore d)': 'shore d hardness iso 48-4 / iso 868 15s',
    'd': 'shore d hardness iso 48-4 / iso 868 15s',
    'hard': 'shore d hardness iso 48-4 / iso 868 15s',
    'shore hardness': 'shore',
    'shore': 'shore',
    'durometer hardness': 'shore',
    'specific heat capacity of melt iso 22007-4 (j/(kg k))': 'specific heat capacity of melt iso 22007-4 (j/(kg k))',
    'surface resistivity iec 62631-3-2 (ohm)': 'surface resistivity iec 62631-3-2 (ohm)',
    'surf_res': 'surface resistivity iec 62631-3-2 (ohm)',
    'sr': 'surface resistivity iec 62631-3-2 (ohm)',
    'tape areal weight - (g/m²)': 'tape areal weight - (g/m²)',
    'tape thickness iso 16012 (mm)': 'tape thickness iso 16012 (mm)',
    'tape width iso 16012 (mm)': 'tape width iso 16012 (mm)',
    'tear strength iso 34-1 normal (kn/m)': 'tear strength iso 34-1 normal (kn/m)',
    'tear stress': 'tear strength iso 34-1 normal (kn/m)',
    'burst strength': 'tear strength iso 34-1 normal (kn/m)',
    'temp. of deflection under load': 'temperature of deflection under load',
    'dtul': 'temperature of deflection under load',
    'hdt': 'temperature of deflection under load',
    'deflection temperature': 'temperature of deflection under load',
    'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'temperature of deflection under load 0.45 mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'temp. of deflection under load 0.45 mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'deflection temperature 0.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'deflection temperature 66psi': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'hdt 0.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'temp. of deflection under load 65 psi annealed': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dtul 0.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'temperature of deflection under load 1.80 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'temp. of deflection under load 1.80 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'deflection temperature 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'deflection temperature 264psi': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'hdt 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'temp. of deflection under load 260 psi annealed': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'dtul 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'temperature of deflection under load iso 75-1/-2 8 mpa (°c)': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'temperature of deflection under load 8.0 mpa': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'temp. of deflection under load 8.0 mpa': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'deflection temperature 8.0mpa': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'hdt 8.0mpa': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'dtul 8.0mpa': 'temperature of deflection under load iso 75-1/-2 8 mpa (°c)',
    'tensile creep modulus iso 899-1 1000h (mpa)': 'tensile creep modulus iso 899-1 1000h (mpa)',
    'tensile creep modulus at 1000h': 'tensile creep modulus iso 899-1 1000h (mpa)',
    'tensile creep modulus iso 899-1 1h (mpa)': 'tensile creep modulus iso 899-1 1h (mpa)',
    'tensile creep modulus at 1h': 'tensile creep modulus iso 899-1 1h (mpa)',
    'young modulus': 'tensile modulus',
    "young's modulus": 'tensile modulus',
    'stiff': 'tensile modulus',
    'ten mod': 'tensile modulus',
    'e-modulus': 'tensile modulus',
    'e modulus': 'tensile modulus',
    'modulus of elasticity': 'tensile modulus',
    'tensile mod': 'tensile modulus',
    "tensile young's modulus": 'tensile modulus',
    'youngs e modulus': 'tensile modulus',
    'youngs modulus': 'tensile modulus',
    'emod': 'tensile modulus',
    'e-mod': 'tensile modulus',
    'tensile module': 'tensile modulus',
    # 'modulus': 'modulus',
    'stiffness': 'tensile modulus',
    'mechanical strength': 'tensile modulus',
    'mechanical behavior': 'tensile modulus',
    'mechanical resistance': 'tensile modulus',
    'mechanical properties': 'tensile modulus',
    'deformation': 'tensile modulus',
    'deformation stability': 'tensile modulus',
    'mechanical performance': 'tensile modulus',
    'tensile': 'tensile',
    'elastic modulus': 'tensile modulus',
    'strength modulus': 'tensile modulus',
    'tensile modulus astm d 3039 m tape 0° (mpa)': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tape tensile mod': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tape modulus of elasticity': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tape e modulus': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'tensile notched impact strength iso 8256/1 23°c (kj/m²)': 'tensile notched impact strength iso 8256/1 23°c (kj/m²)',
    'tensile impact notched': 'tensile notched impact strength iso 8256/1 23°c (kj/m²)',
    'n tenile impact': 'tensile notched impact strength iso 8256/1 23°c (kj/m²)',
    'break strain, break elongation': 'tensile strain at break',
    'strain at break': 'strain at break',
    'elongation break': 'tensile strain at break',
    'tensile strain at break iso 527-1/-2 50mm/min (%)': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'tensile strain at break 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'strain at break 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'strain at break 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'strain break 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'strain break 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'break strain 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'break strain 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'elongation at break 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'elongation at break 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'elongation break 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'elongation break 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'break elongation 50mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'break elongation 50 mm/min': 'tensile strain at break iso 527-1/-2 50mm/min (%)',
    'tensile strain at break iso 527-1/-2 5mm/min (%)': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'tensile strain at break 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'strain at break 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'strain at break 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'strain break 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'strain break 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'break strain 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'break strain 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'elongation at break 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'elongation at break 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'elongation break 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'elongation break 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'break elongation 5mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'break elongation 5 mm/min': 'tensile strain at break iso 527-1/-2 5mm/min (%)',
    'tensile strain at failure astm d 3039 m tape 0° (%)': 'tensile strain at failure astm d 3039 m tape 0° (%)',
    'tensile strain at yield iso 527-1/-2 50mm/min (%)': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'tensile strain at yield 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain at yield 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain at yield 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain yield 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain yield 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield strain 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield strain 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield strain': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain at yield': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'elongation at yield 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'elongation at yield 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'elongation yield 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'elongation yield 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield elongation 50mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield elongation 50 mm/min': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'yield elongation': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'elongation at yield': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'strain yield': 'tensile strain at yield iso 527-1/-2 50mm/min (%)',
    'tensile strength astm d 3039 m tape 0° (mpa)': 'tensile strength astm d 3039 m tape 0° (mpa)',
    'tape tensile strength': 'tensile strength astm d 3039 m tape 0° (mpa)',
    'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at 100% elongation, perpendicular': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at 100% strain, perpendicular': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at 100% deformation, perpendicular': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'perpendicular tensile stress at 100% elongation': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'perpendicular tensile stress at 100% strain': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'perpendicular tensile stress at 100% deformation': 'tensile stress at 100% elongation iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at 100% strain iso 527-1/-2 (mpa)': 'tensile stress at 100% strain iso 527-1/-2 (mpa)',
    'stress at 100% elongation': 'tensile stress at 100% elongation iso 37 perpendicular (mpa)',
    'stress at 100% deformation': 'tensile stress at 100% strain iso 527-1/-2 (mpa)',
    'tensile stress at 50% strain iso 527-1/-2 (mpa)': 'tensile stress at 50% strain iso 527-1/-2 (mpa)',
    'stress at 50% elongation': 'tensile stress at 50% strain iso 527-1/-2 (mpa)',
    'stress at 50% deformation': 'tensile stress at 50% strain iso 527-1/-2 (mpa)',
    'strebr': 'stress at break',
    'tensile strength at break': 'stress at break',
    'tensile stress (break)': 'stress at break',
    'break stress': 'stress at break',
    'ten stress brk': 'stress at break',
    'tensile stress at break iso 527-1/-2 50mm/min (mpa)': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'tensile stress at break at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'strebr at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'tensile strength at break at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'tensile stress (break) at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'break stress at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'ten stress brk at 50mm/min': 'tensile stress at break iso 527-1/-2 50mm/min (mpa)',
    'tensile stress at break iso 527-1/-2 5mm/min (mpa)': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'tensile stress at break at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'strebr at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'tensile strength at break at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'tensile stress (break) at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'break stress at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'ten stress brk at 5mm/min': 'tensile stress at break iso 527-1/-2 5mm/min (mpa)',
    'tensile stress at break iso 527-1/-2 or iso 37 perpendicular (mpa)': 'tensile stress at break iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at break perpendicular': 'tensile stress at break iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'perpendicular stress at break': 'tensile stress at break iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'perpendicular break stress': 'tensile stress at break iso 527-1/-2 or iso 37 perpendicular (mpa)',
    'tensile stress at yield iso 527-1/-2 50mm/min (mpa)': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'tensile stress at yield 50mm/min': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'tensile strength (yield)': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'tensile strength at yield': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'tensile stress (yield)': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'yield stress': 'tensile stress at yield iso 527-1/-2 50mm/min (mpa)',
    'thermal conductive': 'thermal conductivity',
    'thermal shock': 'thermal conductivity',
    'heat transfer': 'thermal conductivity',
    'thermal conductivity iso 22007-2 crossflow (w/(m k))': 'thermal conductivity iso 22007-2 crossflow (w/(m k))',
    'thermal conductivity crossflow direction': 'thermal conductivity iso 22007-2 crossflow (w/(m k))',
    'cross plane thermal conductivity': 'thermal conductivity iso 22007-2 crossflow (w/(m k))',
    'thermal conductivity iso 22007-2 flow (w/(m k))': 'thermal conductivity iso 22007-2 flow (w/(m k))',
    'thermal conductivity flow direction': 'thermal conductivity iso 22007-2 flow (w/(m k))',
    'in plane thermal conductivity': 'thermal conductivity iso 22007-2 flow (w/(m k))',
    'thermal conductivity of melt': 'thermal conductivity iso 22007-2 flow (w/(m k))',
    'thermal conductivity iso 22007-2 through plane (w/(m k))': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'thermal conductivity through plane': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'thermal conductivity thruplane direction': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'thruplane thermal conductivity': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'vicat': 'vicat softening temperature',
    'vicat softening point': 'vicat softening temperature',
    'vicat softing temp': 'vicat softening temperature',
    'vicat softening temperature iso 306 50°c/h 10n (°c)': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
    'vicat softening temperature at 50°c/h 10n': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
    'vicat at 50°c/h 10n': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
    'vicat softening point at 50°c/h 10n': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
    'vicat softing temp at 50°c/h 10n': 'vicat softening temperature iso 306 50°c/h 10n (°c)',
    'vicat softening temperature iso 306 50°c/h 50n (°c)': 'vicat softening temperature iso 306 50°c/h 50n (°c)',
    'vicat softening temperature at 50°c/h 50n': 'vicat softening temperature iso 306 50°c/h 50n (°c)',
    'vicat at 50°c/h 50n': 'vicat softening temperature iso 306 50°c/h 50n (°c)',
    'vicat softening point at 50°c/h 50n': 'vicat softening temperature iso 306 50°c/h 50n (°c)',
    'vicat softing temp at 50°c/h 50n': 'vicat softening temperature iso 306 50°c/h 50n (°c)',
    'viscosity number iso 307, 1628 (cm³/g)': 'viscosity number iso 307, 1628 (cm³/g)',
    'h2so4': 'viscosity number iso 307, 1628 (cm³/g)',
    'viscosity number (0.5% in 96 % h2so4)': 'viscosity number iso 307, 1628 (cm³/g)',
    'vn at 0.5% in sulfuric acid nominal': 'viscosity number iso 307, 1628 (cm³/g)',
    'viscosity h2so4': 'viscosity number iso 307, 1628 (cm³/g)',
    'volume resistivity iec 62631-3-1 (ohm.m)': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'volres': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'vr': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'water absorption sim. to iso 62 2mm (%)': 'water absorption sim. to iso 62 2mm (%)',
    'h20 absorption': 'water absorption sim. to iso 62 2mm (%)',
    'water': 'water absorption sim. to iso 62 2mm (%)',
    'moisture uptake': 'water absorption sim. to iso 62 2mm (%)',
    'wear by sandslurry method': 'wear by sandslurry method (based on gur 4120=100)',
    
    'arc resistance': 'arc resistance',
    'hvar': 'arc resistance',
    'arc resistivity': 'arc resistance',
    'arc resist': 'arc resistance',
    'high voltage arc resist to ignition': 'arc resistance',
    'high voltage arc resistant to ignition': 'arc resistance',
    'high voltage arc resistance to ignition': 'arc resistance',
    'high voltage arc resistivity to ignition': 'arc resistance',
    'high voltage arc resist ignition': 'arc resistance',
    'high voltage arc resistant ignition': 'arc resistance',
    'high voltage arc resistance ignition': 'arc resistance',
    'high voltage arc resistivity ignition': 'arc resistance',
    'high voltage arc': 'arc resistance',
    'arc resistant': 'arc resistance',
    'ar': 'arc resistance',
    'arc r': 'arc resistance',
    'arcres': 'arc resistance',
    'arc res': 'arc resistance',
    'ball pressure test': 'ball pressure test',
    'ball pressure': 'ball pressure test',
    'bpt': 'ball pressure test',
    'ball test': 'ball pressure test',
    'comparative tracking': 'comparative tracking index (cti)',
    'comparative tracking index': 'comparative tracking index (cti)',
    'cti': 'comparative tracking index (cti)',
    'comparative tracking rating': 'comparative tracking index (cti)',
    'comparative tracking index rating': 'comparative tracking index (cti)',
    'comp tracking': 'comparative tracking index (cti)',
    'comparative tracking index (cti)': 'comparative tracking index (cti)',
    'iec tracking index': 'comparative tracking index (cti)',
    'tracking index': 'comparative tracking index (cti)',
    'dielectric strength': 'dielectric strength',
    'di strength': 'dielectric strength',
    'dielectric str': 'dielectric strength',
    'di str': 'dielectric strength',
    'diele str': 'dielectric strength',
    'dielec str': 'dielectric strength',
    'dielectrical strength': 'dielectric strength',
    'dc dielectric strength': 'dielectric strength',
    'di-electric str': 'dielectric strength',
    'electrically insulative': 'dielectric strength',
    'electrical insulation': 'dielectric strength',
    'dielectric breakdown strength': 'dielectric strength',
    'dimensional change': 'dimensional change',
    'dim change': 'dimensional change',
    'dimens change': 'dimensional change',
    'dimensional chg': 'dimensional change',
    'di-mensional change': 'dimensional change',
    'high voltage arc tracking rate': 'high voltage arc tracking rate (hvtr)',
    'hvtr': 'high voltage arc tracking rate (hvtr)',
    'high vol arc tracking rate': 'high voltage arc tracking rate (hvtr)',
    'high voltage tracking rate': 'high voltage arc tracking rate (hvtr)',
    'high voltage arc tracking': 'high voltage arc tracking rate (hvtr)',
    'voltage arc tracking rate': 'high voltage arc tracking rate (hvtr)',
    'high voltage rate': 'high voltage arc tracking rate (hvtr)',
    'high voltage arc tracking rate (hvtr)': 'high voltage arc tracking rate (hvtr)',
    'inclined-plane': 'inclined-plane tracking',
    'ip tracking': 'inclined-plane tracking',
    'ipt': 'inclined-plane tracking',
    'inclined plane': 'inclined-plane tracking',
    'plane tracking': 'inclined-plane tracking',
    'inc-plane track': 'inclined-plane tracking',
    'inc-plane tracking': 'inclined-plane tracking',
    'inclined plane tracking': 'inclined-plane tracking',
    'non-halogenated material': 'non-halogenated material',
    'nhm': 'non-halogenated material',
    'non halo material': 'non-halogenated material',
    'outdoor suitability': 'outdoor suitability',
    'outdoor durability': 'outdoor suitability',
    'out door durability': 'outdoor suitability',
    # 'durability': 'outdoor suitability',
    # 'suitability': 'outdoor suitability',
    'out door suitability': 'outdoor suitability',
    'os': 'outdoor suitability',
    'detergent resistance': 'detergent resistance',
    'detergent resistant': 'detergent resistance',
    'detergent resist': 'detergent resistance',
    'detergent resistivity': 'detergent resistance',
    'detergent resis': 'detergent resistance',
    'detergent res': 'detergent resistance',
    'deterres': 'detergent resistance',
    'dr': 'detergent resistance',
    'dres': 'detergent resistance',
    'detergence resistance': 'detergent resistance',
    'rohs': 'rohs 2011/65/eu material',
    'rohs and eu': 'rohs 2011/65/eu material',
    'rohs & eu': 'rohs 2011/65/eu material',
    'rohs 2011/65/eu material': 'rohs 2011/65/eu material',
    'rohs material': 'rohs 2011/65/eu material',
    '2011/65/eu material': 'rohs 2011/65/eu material',
    '2011/65/eu': 'rohs 2011/65/eu material',
    'rohs 2011/65/eu mat': 'rohs 2011/65/eu material',
    'rohs mat': 'rohs 2011/65/eu material',
    '2011/65/eu mat': 'rohs 2011/65/eu material',
    'tensile impact strength': 'tensile impact strength',
    'ten imp str': 'tensile impact strength',
    'tensile imp': 'tensile impact strength',
    'tensile imp str': 'tensile impact strength',
    'ti strength': 'tensile impact strength',
    'ti str': 'tensile impact strength',
    'tis': 'tensile impact strength',
    
    'flame rating': 'flame rating',
    'fr': 'flame rating',
    'flame rated': 'flame rating',
    'flame rate': 'flame rating',
    'flame': 'flame rating',
    'flamerating': 'flame rating',
    'flamerated': 'flame rating',
    'flamerate': 'flame rating',
    'ul listed': 'flame rating',
    'flame class': 'flammability classification',
    'flame classification': 'flammability classification',
    'flammability': 'flammability',
    'fc': 'flammability classification',
    'flameclassification': 'flammability classification',
    'flammability classification': 'flammability classification',
    'glow wire flammability index': 'glow wire flammability index',
    'glow-wire flammability': 'glow wire flammability index',
    'gwfi': 'glow wire flammability index',
    'glow wire flammability': 'glow wire flammability index',
    'glow-wire flammability index': 'glow wire flammability index',
    'glow wire test': 'glow wire ignition temperature',
    'gwf index': 'glow wire flammability index',
    'glow wire flame index': 'glow wire flammability index',
    'glow wire ignition': 'glow wire ignition temperature',
    'gwit': 'glow wire ignition temperature',
    'glow wire temperature': 'glow wire ignition temperature',
    'glow wire temp': 'glow wire ignition temperature',
    'glow-wire ignition temperature': 'glow wire ignition temperature',
    'gwi temp': 'glow wire ignition temperature',
    'gwi temperature': 'glow wire ignition temperature',
    'flow wire ignition temperature': 'glow wire ignition temperature',
    'glow wire': 'glow wire',
    'high-current arc ignition': 'high-current arc ignition (hai)',
    'high current arc ignition': 'high-current arc ignition (hai)',
    'high amp arc ignition': 'high-current arc ignition (hai)',
    'high ampere arc ignition': 'high-current arc ignition (hai)',
    'high amperage arc ignition category': 'high-current arc ignition (hai)',
    'hai': 'high-current arc ignition (hai)',
    'high current arc': 'high-current arc ignition (hai)',
    'high arc ignition': 'high-current arc ignition (hai)',
    'arc ignition': 'high-current arc ignition (hai)',
    'hcai': 'high-current arc ignition (hai)',
    'high-current ignition': 'high-current arc ignition (hai)',
    'current ignition': 'high-current arc ignition (hai)',
    'high amp': 'high-current arc ignition (hai)',
    'high current arc ignition (hai)': 'high-current arc ignition (hai)',
    'hot wire ignition': 'hot wire ignition (hwi)',
    'hwi': 'hot wire ignition (hwi)',
    'hot-wire ignition': 'hot wire ignition (hwi)',
    'hw ignition': 'hot wire ignition (hwi)',
    'hot wire': 'hot wire ignition (hwi)',
    'hot wire ignition (hwi)': 'hot wire ignition (hwi)',
    'relative thermal index electrical': 'relative thermal index - electrical (rti elec) (°c)',
    'relative thermal index ele': 'relative thermal index - electrical (rti elec) (°c)',
    'rti-e': 'relative thermal index - electrical (rti elec) (°c)',
    'rti elec': 'relative thermal index - electrical (rti elec) (°c)',
    'rti ele': 'relative thermal index - electrical (rti elec) (°c)',
    'rti electrical': 'relative thermal index - electrical (rti elec) (°c)',
    'rtie': 'relative thermal index - electrical (rti elec) (°c)',
    'relative thermal index - electrical (rti elec)': 'relative thermal index - electrical (rti elec) (°c)',
    'relative thermal index - electrical': 'relative thermal index - electrical (rti elec) (°c)',
    'relative thermal index mechanical impact': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'relative thermal index impact': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'rti-i': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'rti imp': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'rti impact': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'rtii': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'relative thermal index - mechanical impact (rti imp)': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'relative thermal index - mechanical impact': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'relative thermal index mechanical strength': 'relative thermal index - mechanical strength (rti str) (°c)',
    'relative thermal index strength': 'relative thermal index - mechanical strength (rti str) (°c)',
    'rti-s': 'relative thermal index - mechanical strength (rti str) (°c)',
    'rti str': 'relative thermal index - mechanical strength (rti str) (°c)',
    'rti strength': 'relative thermal index - mechanical strength (rti str) (°c)',
    'relative thermal index mechanical': 'relative thermal index - mechanical strength (rti str) (°c)',
    'rti ms': 'relative thermal index - mechanical strength (rti str) (°c)',
    'relative thermal index - mechanical strength (rti str)': 'relative thermal index - mechanical strength (rti str) (°c)',
    'relative thermal index - mechanical strength': 'relative thermal index - mechanical strength (rti str) (°c)',
    'relative thermal index': 'relative thermal index',
    'rti': 'relative thermal index',
    
    
    # survey queries
    'emodulus': 'tensile modulus',
    'mt': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'thermally conduct.': 'thermal conductivity',
    'charpy impact strenght': 'charpy impact strength',
    'coefficient of liner thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'brittleness temp': 'brittleness temperature',
    'service temp': 'continuous service temperature iec 60216-1 (°c)',
    'flex strength at yield': 'flexural strength at yield',
    'charpy impact strength ( notched )': 'charpy notched impact strength',
    'unnotched charpy impact strength': 'charpy impact strength',
    'dtul c': 'temperature of deflection under load',
    'compression modulus': 'compressive modulus iso 604 (mpa)',
    'dielectric constant': 'relative permittivity',
    'shore d hardness': 'shore d hardness iso 48-4 / iso 868 15s',
    'heat deflection temp at .45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'tc ( thruplane )': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'flexral modulus': 'flexural modulus',
    'high voltage arc tracking rate ( hvtr )': 'high voltage arc tracking rate (hvtr)',
    'modules': 'modulus',
    'deflection temperature under load ( 1.8mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'dielectric constant (dk) dielectric constant': 'relative permittivity',
    'tesile modulus': 'tensile modulus',
    'heat deflection': 'temperature of deflection under load',
    'dtul @1,8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'puncture maximum force iso 6603-2 23°c (n)': 'puncture - maximum force iso 6603-2 23°c (n)',
    'cm': 'compressive modulus iso 604 (mpa)',
    "young's modulus ( tension )": 'tensile modulus',
    'dtul @ 0.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    # 'specific gravity': 'bulk density iso 60 (kg/m³)',
    'water absorp @ 23c 50% rh': 'water absorption sim. to iso 62 2mm (%)',
    "young's modulus (tension)": 'tensile modulus',
    'service temperatures': 'continuous service temperature iec 60216-1 (°c)',
    'loss tangent ( 0.0001 ) at 10ghz': 'dissipation factor iec 62631-2-1 10ghz (e-4)',
    'rohs 2011/65/eu': 'rohs 2011/65/eu material',
    'elastic modulus (tension)': 'tensile modulus',
    'melt point': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    # 'thermal shock resistance': 'thermal conductivity',
    'dtul @ 1,8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'surface resisitivity ( 23c )': 'surface resistivity iec 62631-3-2 (ohm)',
    'ncharpy': 'charpy notched impact strength',
    'glas transition temperature': 'glass transition temperature iso 11357-1/-3 10°c/min (°c)',
    'diel. constant': 'relative permittivity',
    'deflection temparature': 'temperature of deflection under load',
    'deflection temperature under load ( dtul )': 'temperature of deflection under load',
    'modulus of compression': 'compressive modulus iso 604 (mpa)',
    'tensile modules': 'tensile modulus',
    'tc': 'thermal conductivity',
    'surf resis': 'surface resistivity iec 62631-3-2 (ohm)',
    'heat deflection temp .45 mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dtul ( at 1.8mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melting temp': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'dtul (1.80 mpa) unannealed': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'tensile md': 'tensile modulus',
    'passion ratio': "poisson's ratio",
    'density': 'density iso 1183 (kg/m³)',
    'coefficient of linear thermal expansion ( parallel )': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'iso tensile modulus': 'tensile modulus iso 527-1/-2 (mpa)',
    'durometer': 'shore',
    'dtul (1.8mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'coeff. of linear therm expansion ( parallel )': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'cont service temp': 'continuous service temperature iec 60216-1 (°c)',
    'coeff. of linear therm expansion parallel': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'heat deflection temp ( .45 mpa )': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'nc impact': 'charpy notched impact strength',
    'thermal conductivity , thruplane': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'tensile modus': 'tensile modulus',
    'deflection temp under load ( 1.8mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'impact strenght': 'impact strength',
    'electric strength ( ac )': 'electric strength iec 60243-1 (kv/mm)',
    'dielectric stregth': 'dielectric strength',
    'cont. service temp': 'continuous service temperature iec 60216-1 (°c)',
    'mold shringkage': 'molding shrinkage',
    'tear strenght': 'tear strength iso 34-1 normal (kn/m)',
    'dt loaded 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'charpy strength': 'charpy impact strength',
    'charpy impact str': 'charpy impact strength',
    'melt volume-flow rate': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'dielectric strength ( ac )': 'electric strength iec 60243-1 (kv/mm)',
    'deflection temperature under load (1.8mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'elastic moduls': 'tensile modulus',
    'coeffieent of linear thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'charpy impact resistance unnotched': 'charpy impact strength',
    'melt flow': 'melt mass-flow rate iso 1133 (g/10min)',
    'dtul at 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'mfi': 'melt mass-flow rate iso 1133 (g/10min)',
    'ac electric strength': 'electric strength iec 60243-1 (kv/mm)',
    'tensile stiffness': 'tensile modulus',
    'mvtr': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'dtul@1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    # 'ms': 'relative thermal index - mechanical strength (rti str) (°c)', # since unit is %, user meant Molding shrinkage
    'equilibrium water absorption at 23c/50% rh': 'water absorption sim. to iso 62 2mm (%)',
    'water absortion': 'water absorption sim. to iso 62 2mm (%)',
    'flexural modules': 'flexural modulus',
    'unnotched impact resistance': 'charpy impact strength',
    'equilibrium water absorption (23c, 50% rh)': 'water absorption sim. to iso 62 2mm (%)',
    'partical size': 'average particle size laser scattering d50 (µm)',
    'deflection temp under load (1.8mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'iso 527 tensile modulus': 'tensile modulus iso 527-1/-2 (mpa)',
    'flex. mod': 'flexural modulus',
    'equilibrium water absorption at 23c / 50% rh': 'water absorption sim. to iso 62 2mm (%)',
    'wear absorption': 'water absorption sim. to iso 62 2mm (%)',
    'ps(avg)': 'average particle size laser scattering d50 (µm)',
    'inclined-plane tracking ( astm d2303 )': 'inclined-plane tracking',
    'electric str': 'electric strength iec 60243-1 (kv/mm)',
    'temperature of embrittlement': 'brittleness temperature',
    'water absorption ( equilibrium, 23c, 50% relative humidity )': 'water absorption sim. to iso 62 2mm (%)',
    'flex strength': 'flexural strength',
    'flexural stress': 'flexural stress',
    'clte-parallel': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'shrink': 'molding shrinkage',
    'dtul @ 1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melting': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'thermal cond.': 'thermal conductivity',
    'flamability rating': 'flame rating',
    'molecluar weight': "average molecular weight margolies' equation (g/mol)",
    'electrical resistivity ( surface )': 'surface resistivity iec 62631-3-2 (ohm)',
    'tensile creep modulus, 1000h': 'tensile creep modulus iso 899-1 1000h (mpa)',
    'continuous use environment': 'continuous service temperature iec 60216-1 (°c)',
    'heat deflection temp': 'temperature of deflection under load',
    'heat deflection at 1.8 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'thermally conductive through plane': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'impact strength-charpy': 'charpy impact strength',
    'dielectric properties': 'dielectric strength',
    'mold': 'molding shrinkage',
    'flexurat modulus': 'flexural modulus',
    'brittleness point': 'brittleness temperature',
    'coefficient of linear thermal expansion(parallel)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    '1.8 dtul': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'dtul at 1.8 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'ball hardness': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'sresis': 'surface resistivity iec 62631-3-2 (ohm)',
    'continuous tempareture': 'continuous service temperature iec 60216-1 (°c)',
    'water absorption (equilibrium, 23c, 50% relative humidity)': 'water absorption sim. to iso 62 2mm (%)',
    'particle size': 'average particle size laser scattering d50 (µm)',
    'acceptable shrinkage': 'molding shrinkage',
    'temp': 'continuous service temperature iec 60216-1 (°c)',
    'pressure': 'ball pressure test',
    'melt mass-flow rate': 'melt mass-flow rate iso 1133 (g/10min)',
    'vol. resis.': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'volume res': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'water immersion': 'water absorption sim. to iso 62 2mm (%)',
    'tensile modulus tape': 'tensile modulus astm d 3039 m tape 0° (mpa)',
    'water absorp @23c 50%rh': 'water absorption sim. to iso 62 2mm (%)',
    'charpy high impact strength': 'charpy impact strength',
    'charphy imact': 'charpy impact strength',
    'vol resistivity': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'inclined-plane tracking 2.5kv (astm d2303)': 'inclined-plane tracking',
    'e ( ac )': 'electric strength iec 60243-1 (kv/mm)',
    'continuous service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'molecular weigth': "average molecular weight margolies' equation (g/mol)",
    'water absorp.': 'water absorption sim. to iso 62 2mm (%)',
    'electric strength (ac)': 'electric strength iec 60243-1 (kv/mm)',
    'surface resistance': 'surface resistivity iec 62631-3-2 (ohm)',
    'dtul @1.8mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'mold shrink': 'molding shrinkage',
    'shrinkage': 'molding shrinkage',
    'dtul 1.8 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'ds(ac)': 'electric strength iec 60243-1 (kv/mm)',
    'flexural module': 'flexural modulus',
    'mould shrinage': 'molding shrinkage',
    'volume r': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'tensible modulus': 'tensile modulus',
    'volume resitivity': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'water absoption': 'water absorption sim. to iso 62 2mm (%)',
    'dtul 1.8': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'mol. wt.': "average molecular weight margolies' equation (g/mol)",
    'through plane thermal conductivity': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'brittleness': 'brittleness temperature',
    'elasticity': 'tensile modulus',
    'elongation': 'elongation',
    'dtul ( 1.8mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'brittle temp': 'brittleness temperature',
    'comparative tracking index (cti) (ul 746a)': 'comparative tracking index (cti)',
    'impact strength - charpy': 'charpy impact strength',
    'ctle': 'coefficient of linear thermal expansion (clte)',
    'equilibrium water absorption ( 23°c/50% rh )': 'water absorption sim. to iso 62 2mm (%)',
    'use temperature': 'continuous service temperature iec 60216-1 (°c)',
    'compression e': 'compression set',
    'tmelt': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'tensile modulus of elasticity': 'tensile modulus',
    'water absorption at equilibrium, 23 deg, 50% rh': 'water absorption sim. to iso 62 2mm (%)',
    'arc resis': 'arc resistance',
    'elongation (strain at break)': 'tensile strain at break',
    'dielectric': 'dielectric strength',
    'ps ( avg )': 'average particle size laser scattering d50 (µm)',
    'absorb': 'water absorption sim. to iso 62 2mm (%)',
    'electr.strength': 'electric strength iec 60243-1 (kv/mm)',
    'compressive mod': 'compressive modulus iso 604 (mpa)',
    'continous usage temperature': 'continuous service temperature iec 60216-1 (°c)',
    "charpy's impact ( unnotched )": 'charpy impact strength',
    'hdt/1.8': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'impact strength': 'impact strength',
    'heat deflection temperature': 'temperature of deflection under load',
    'dtul at 0.45 mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dtul ( 1.8 mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'continuous service temp': 'continuous service temperature iec 60216-1 (°c)',
    'dimensional stability': 'dimensional change',
    'specific resitivity': 'specific resistivity',
    'shrinkage ( flow )': 'molding shrinkage iso 294-4, 2577 parallel (%)',
    'avg particle size': 'average particle size laser scattering d50 (µm)',
    'equilibrium water absorption (23°c/50%rh)': 'water absorption sim. to iso 62 2mm (%)',
    'creep': 'tensile creep modulus',
    'shore a hardness, 15s': 'shore a hardness iso 48-4 / iso 868 15s',
    'ball intention hardness': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'ball indent hardness': 'ball indentation hardness iso 2039-1 h 358/30 (mpa)',
    'unnotched impact strength': 'charpy impact strength',
    'dsc melting t': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'melt temperature': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'continurous service temperature': 'continuous service temperature iec 60216-1 (°c)',
    'elec strength (ac)': 'electric strength iec 60243-1 (kv/mm)',
    'coeff. of linear therm expansion (parallel)': 'coefficient of linear thermal expansion (clte) iso 11359-1/-2 parallel (e-6/k)',
    'flexular modulus': 'flexural modulus',
    'deflection temperature under load ( 1.8 mpa )': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'aps': 'average particle size laser scattering d50 (µm)',
    'avg part size': 'average particle size laser scattering d50 (µm)',
    'inclided plane tracking': 'inclined-plane tracking',
    'mvfr': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'dtul ( 1.80 mpa ) unannealed': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melting temp. melting temperature': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'average powder size': 'average particle size laser scattering d50 (µm)',
    'water absorption ( equilibrium, 23 degrees celsius, 50% ph )': 'water absorption sim. to iso 62 2mm (%)',
    'electrical resistivity (surface)': 'surface resistivity iec 62631-3-2 (ohm)',
    'limiting oxygen index (loi)': 'oxygen index iso 4589-1/-2 (%)',
    'thruplane': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    # 'electric properties': 'electric strength iec 60243-1 (kv/mm)',
    'melt temperatures': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'vresis': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'hdt@0.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dtul(1.8mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melting point': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'water absorption (equilibrium, 23 deg c, 50&% rh)': 'water absorption sim. to iso 62 2mm (%)',
    'dimension stable': 'dimensional change',
    'equilibrium water absorption ( 23c, 50% rh )': 'water absorption sim. to iso 62 2mm (%)',
    'vol res': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'water absorption (equilibrium, 23 degrees celsius, 50%ph)': 'water absorption sim. to iso 62 2mm (%)',
    'n charpy': 'charpy notched impact strength',
    'elec strength ( ac )': 'electric strength iec 60243-1 (kv/mm)',
    'tensil modulus': 'tensile modulus',
    'shrink.': 'molding shrinkage',
    'elastic modulus ( tension )': 'tensile modulus',
    'relative thermal index – electrical (rti elec) (°c)': 'relative thermal index - electrical (rti elec) (°c)',
    'dielectric constant (dk)': 'relative permittivity',
    'ten. mod': 'tensile modulus',
    'relative thermal index – mechanical impact (rti imp) (°c)': 'relative thermal index - mechanical impact (rti imp) (°c)',
    'coeff. of linear therm expansion': 'coefficient of linear thermal expansion (clte)',
    'relative thermal index – mechanical strength (rti str) (°c)': 'relative thermal index - mechanical strength (rti str) (°c)',
    'dtul - 1.8 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'flxural strength': 'flexural strength',
    'shore a hardness': 'shore a hardness iso 48-4 / iso 868 15s',
    'unnotched izod impact strength': 'izod impact strength',
    'shrink rate': 'molding shrinkage',
    'deflection temperature under load (1.8 mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'compressive modules': 'compressive modulus iso 604 (mpa)',
    'tmod': 'tensile modulus',
    'vol resist': 'volume resistivity iec 62631-3-1 (ohm.m)',
    'deflection temp': 'temperature of deflection under load',
    'dtul @ o.45mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dtul (1.8 mpa)': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melt volume-flow': 'melt volume-flow rate iso 1133 (cm³/10min)',
    'thermal conductivity, thruplane': 'thermal conductivity iso 22007-2 through plane (w/(m k))',
    'cst': 'continuous service temperature iec 60216-1 (°c)',
    'electrical resistivity': 'surface resistivity iec 62631-3-2 (ohm)',
    'water absorption ( equilibrium, 23 deg c, 50 & rh )': 'water absorption sim. to iso 62 2mm (%)',
    'charpy notched impact strength, 23°c': 'charpy notched impact strength iso 179/1ea 23°c (kj/m²)',
    'e(ac)': 'electric strength iec 60243-1 (kv/mm)',
    'deflection temperature under load (dtul)': 'temperature of deflection under load',
    'continuous temperature': 'continuous service temperature iec 60216-1 (°c)',
    'coeffiecient of linear thermal expansion': 'coefficient of linear thermal expansion (clte)',
    'heat deflection temperature at 1.8 mpa': 'temperature of deflection under load iso 75-1/-2 1.8 mpa (°c)',
    'melting temperature, 10°c/min': 'melting temperature iso 11357-1/-3 10°c/min (°c)',
    'elongtion': 'elongation',
    'water absorption': 'water absorption sim. to iso 62 2mm (%)',
    'den': 'density iso 1183 (kg/m³)',
    'flex str.': 'flexural strength',
    "charpy's impact (unnotched)": 'charpy impact strength',
    'mol wt': "average molecular weight margolies' equation (g/mol)",
    'heat deflection temp @ 0.45 mpa': 'temperature of deflection under load iso 75-1/-2 0.45 mpa (°c)',
    'dielectric contant': 'relative permittivity',
    'flexular strength': 'flexural strength',
    'charpy unnotched impact strength': 'charpy impact strength',
    'cof': 'cof',
    'diss. factor': 'dissipation factor',
    'specifiv resistivity': 'specific resistivity', 
    'apparent viscocity': 'apparent viscosity',
    
    'specific gravity': 'density iso 1183 (kg/m³)',
    'clti': 'clte',
    'cof': 'coefficient of friction',
    'fmvss': 'fmvss',
    'unnotched': 'charpy impact strength',
    'glow wire test':  'glow wire',
    'compressive load': 'compressive strength iso 604 (mpa)',
    'rv': 'relative viscosity',
    'tensile stress': 'tensile stress',
    'ltb': 'low temperature brittleness',
    'melt viscosity': 'melt viscosity',
    'melt strength': 'melt strength',
    
    'fogging': 'fogging',
    'average electric strength': 'electric strength iec 60243-1 (kv/mm)',
    'fiber areal weight (g/m²)': 'fiber areal weight - (g/m²)',
    'hardness': 'hardness',
    'strain': 'strain',
    'tensile strain': 'tensile strain',
    'water absorption sim. to iso 62 1mm (%)': 'water absorption sim. to iso 62 2mm (%)',
    'resistivity': 'resistivity',
    'compressive': 'compressive',
    'electrical strength': 'electric strength iec 60243-1 (kv/mm)',
    'ts': 'ts',
    'tape areal weight (g/m²)': 'tape areal weight - (g/m²)',
    'strength': 'tensile stress',
    'minimum thickness (mm)': 'minimum thickness (mm)',
    'heat dissipation': 'thermal conductivity',
    'flammability rating': 'flame rating',
    'temperature resistance': 'continuous service temperature iec 60216-1 (°c)',
    'flame class rating': 'flame rating',
    'elongation at break': 'tensile strain at break',
    'stress at break iso 527-1/-2 (mpa)': 'stress at break iso 527-1/-2 (mpa)',
    
    'impact n': 'notched impact',
    'n charpy': 'charpy notched impact strength',
    'impact (nc)': 'charpy notched impact strength',
    'impact - nc': 'charpy notched impact strength',
    'impact nc': 'charpy notched impact strength',
    'impact - n charpy': 'charpy notched impact strength',
    'impact n charpy': 'charpy notched impact strength',
    
    'nc impact izod': 'izod notched impact strength',
    'nc izod': 'izod notched impact strength',
    'izod (nc)': 'izod notched impact strength',
    'izod impact (nc)': 'izod notched impact strength',
    'izod nc': 'izod notched impact strength',
    'izod - nc': 'izod notched impact strength',
    'nc izod impact': 'izod notched impact strength',
    'melt': 'melt',
    'module elastic': 'tensile modulus',
}


ignore_property = [
    'crush test',
    'abrasion resistance',
    'crystallinity',
    'stress-strain',     
    'pellet size',
    'shear',
    'fmvss',
]

In [14]:
filler_syn_mapping = {
    'aramide fiber': 'aramid fiber',
    'aramide fibre': 'aramid fiber',
    'aramid fiber': 'arami fiber',
    'aramid fibre': 'aramid fiber',
    'af': 'aramid fiber',
    'aramide reinforced': 'aramid fiber',
    'aramid reinforced': 'aramid fiber',
    'carbon fiber': 'carbon fiber',
    'carbon fibre': 'carbon fiber',
    'cf': 'carbon fiber',
    'carbon filled': 'carbon fiber',
    'carbon reinforced': 'carbon fiber',
    'carbon powder': 'carbon powder',
    'cd': 'carbon powder',
    'glass beads': 'glass beads',
    'gb': 'glass beads',
    'glass sphere': 'glass beads',
    'glass balls': 'glass beads',
    'glass fiber': 'glass fiber',
    'glass': 'glass fiber',
    'glass filled': 'glass fiber',
    'fiber': 'glass fiber',
    'fibre': 'glass fiber',
    'gf': 'glass fiber',
    'glass reinforced': 'glass fiber',
    'reinforced': 'glass fiber',
    'gr': 'glass fiber',
    'glas fiber': 'glass fiber',
    'glass fibre': 'glass fiber',
    'glas fibre': 'glass fiber',
    'short fiber': 'glass fiber',
    'short fibre': 'glass fiber',
    'glass reinforcement': 'glass fiber',
    'glass content': 'glass fiber',
    'fiberglass': 'glass fiber',
    'fibreglass': 'glass fiber',
    'fiber glass': 'glass fiber',
    'fibre glass': 'glass fiber',
    'long aramide fiber': 'long aramid fiber',
    'laf': 'long aramid fiber',
    'long aramide fibre': 'long aramid fiber',
    'long aramid fiber': 'long aramid fiber',
    'long aramid fibre': 'long aramid fiber',
    'long carbon fiber': 'long carbon fiber',
    'lcf': 'long carbon fiber',
    'long carbon fibre': 'long carbon fiber',
    'long glass fiber': 'long glass fiber',
    'lgf': 'long glass fiber',
    'long fiber': 'long glass fiber',
    'continuous fiber': 'long glass fiber',
    'cft': 'long glass fiber',
    'long glass fibre': 'long glass fiber',
    'long fibre': 'long glass fiber',
    'long fiberglass': 'long glass fiber',
    'long fibreglass': 'long glass fiber',
    'long fiber glass': 'long glass fiber',
    'long fibre glass': 'long glass fiber',
    'long metal fiber': 'long metal fiber',
    'stainless steel fiber': 'long metal fiber',
    'ss fiber': 'long metal fiber',
    'stainless steel': 'long metal fiber',
    'ss': 'long metal fiber',
    'long steel': 'long metal fiber',
    'long steel fiber': 'long metal fiber',
    'long stainless steel': 'long metal fiber',
    'long stainless steel fiber': 'long metal fiber',
    'long ss fiber': 'long metal fiber',
    'long ss': 'long metal fiber',
    'steel fiber': 'long metal fiber',
    'long metal fibre': 'long metal fiber',
    'sf': 'long metal fiber',
    'metal': 'metal',
    'conductive filler': 'metal',
    'mineral': 'mineral',
    'min': 'mineral',
    'talc': 'mineral',
    'td': 'mineral',
    'mi': 'mineral',
    'md': 'mineral',
    'whisker': 'whisker',
    'un filled': 'unfilled',
    'un-filled': 'unfilled',
    'unfilled': 'unfilled',
    'not filled': 'unfilled',
    'no filler': 'unfilled',
    'non-filler': 'unfilled',
    'non filler': 'unfilled',
    'non filled': 'unfilled',
    'no fill': 'unfilled',
    'without fill': 'unfilled',
    'without filler': 'unfilled',
    'without filled': 'unfilled',
    'no load': 'unfilled',
    'unreinforced': 'unfilled',
    'without glass fiber': 'unfilled',
    'without glass': 'unfilled',
    'without glass fibre': 'unfilled',

    ##
    'graphite fiber': 'graphite fiber',
    'calcium carbonate': 'mineral',
    'carbon': 'carbon',
}

In [15]:
cat_values_mapping = {
    'f 4': 'f4',
    'vo': 'v-0',
    'f 1': 'f1',
    'plc-7': 'plc 7',
    'plc-6': 'plc 6',
    'plc1': 'plc 1',
    '863 compliant': '+863 compliant',
    '+863-compliant': '+863 compliant',
    'plc-0': 'plc 0',
    'plc-3': 'plc 3',
    'f-1': 'f1',
    'plc3': 'plc 3',
    'v2': 'v-2',
    'plc2': 'plc 2',
    '5-va': '5va',
    'plc-1': 'plc 1',
    'plc0': 'plc 0',
    '863-compliant': '+863 compliant',
    'f 3': 'f3',
    'zero': '0',
    'f-2': 'f2',
    'plc-4': 'plc 4',
    'plc4': 'plc 4',
    'hb 40': 'hb40',
    'f-4': 'f4',
    'f 2': 'f2',
    '5-vb': '5vb',
    'v0': 'v-0',
    'v1': 'v-1',
    'v 2': 'v-2',
    'v 0': 'v-0',
    'f-3': 'f3',
    'plc-2': 'plc 2',
    'hb-40': 'hb40',
    '5 vb': '5vb',
    'hb 75': 'hb75',
    'plc6': 'plc 6',
    'v 1': 'v-1',
    'hb-75': 'hb75',
    'plc7': 'plc 7',
    'plc5': 'plc 5',
    '5 va': '5va',
    'v-o': 'v-0',
    'plc-5': 'plc 5',
    'no break': 'nb',
    'nobreak': 'nb',
    'no-break': 'nb',
    'no - break': 'nb',
}

In [16]:
ignore_query_with_apps = [
    # 'sheet and profile',
    # 'powder coating',
    # 'lightweight thermoplastic',
    # 'cooling produce (vegetables, fruits) immediately after harvest',
    # 'static build during transportation',
    # 'shrink tapes',
    # 'outdoor weathering',
    # 'medical application',
    # 'extrusion coating',
    # 'plastic for separator li-ion cell',
    # 'heat dissipation',
    # # 'medical implant', # not sure
    # 'release',
    # 'weather',
    # 'aluminium for profiles', # not sure
    # # 'window profiles', # not sure
    # # 'profiles',
    # 'excellent heat resis',
    # 'lubricated',
    # # 'implanted medical devices', # not sure
    # 'profile for bus',
    # 'sheets',
    # 'medical',
    # 'light weighting',
    # 'melt blown fiber',
    # 'metal replacement bike frame', # not sure
    # 'cable extrusion', # not sure
    # 'autoinjectors that is a good sliding partner',
    # # 'medical devices', # not sure
    # 'fibers',
    # 'glass run channel, inner & outer belt line seals (semi-dynamic)', # two apps
]

In [17]:
outputs = []
features_list = []
del_forms_list = []
brands_list = []
polymers_list = []
processings_list = []
fillers_list = []
properties_list = []
cat_values_list = []
oems_list = []
ignore_rows = []
for idx, row in df.iterrows():
    # print(row['Query'], "\n")
    # print(json.dumps(eval(row['Output']), indent=3), "\n\n")
    output = row['Output']
    # try:  
    if output['FEATURE']:
        output['FEATURE'] = [x.lower() for x in output['FEATURE']]
        features = []
        for f in output['FEATURE']:
            if f=='pelltets':
                output['DELIVERY_FORM'].append('pellets')
                continue
            elif f=='profile' :
                output['PROCESSING'].append('profile extrusion')
                continue

            if f in list(feature_syn_mapping):
                # print(f"{idx}: {f}")
                features.append(feature_syn_mapping[f])
                # print(features, "\n\n")
                if feature_syn_mapping[f] not in features_list:
                    features_list.append(feature_syn_mapping[f])
            else:
                features.append(f)
                if f not in features_list:
                    features_list.append(f)
                            
        output['FEATURE'] = [f.replace('good', '').strip() for f in features if f not in ignore_feature]
        
    if output['DELIVERY_FORM']:
        output['DELIVERY_FORM'] = [x.lower() for x in output['DELIVERY_FORM']]
        del_forms = []
        for item in output['DELIVERY_FORM']:
            if item in list(delivery_syn_mapping):
                # print(f"{idx}: {item}")
                del_forms.append(delivery_syn_mapping[item])
                # print(del_forms, "\n\n")
                if delivery_syn_mapping[item] not in del_forms_list:
                    del_forms_list.append(delivery_syn_mapping[item])
            else:
                del_forms.append(item)
                if item not in del_forms_list:
                    del_forms_list.append(item)
                    
        output['DELIVERY_FORM'] = del_forms

    if output['BRAND']:
        output['BRAND'] = [x.lower() for x in output['BRAND']]
        brands = []
        for item in output['BRAND']:
            if item in list(brand_syn_mapping):
                # print(f"{idx}: {item}")
                brands.append(brand_syn_mapping[item])
                # print(brands, "\n\n")
                if brand_syn_mapping[item] not in brands_list:
                    brands_list.append(brand_syn_mapping[item])
            else:
                brands.append(item)
                if item not in brands_list:
                    brands_list.append(item)
                    
        output['BRAND'] = [i for i in brands if i not in ignore_brand]

    if output['POLYMER']:
        output['POLYMER'] = [x.lower() for x in output['POLYMER']]
        polymers = []
        for item in output['POLYMER']:
            if item in list(polymer_syn_mapping):
                # print(f"{idx}: {item}")
                polymers.append(polymer_syn_mapping[item])
                # print(polymers, "\n\n")
                if polymer_syn_mapping[item] not in polymers_list:
                    polymers_list.append(polymer_syn_mapping[item])
            else:
                polymers.append(item)
                if item not in polymers_list:
                    polymers_list.append(item)
                    
        output['POLYMER'] = [i for i in polymers if i not in ignore_polymers]

    if output['PROCESSING']:
        output['PROCESSING'] = [x.lower() for x in output['PROCESSING']]
        processings = []
        for item in output['PROCESSING']:
            if item in list(processing_syn_mapping):
                # print(f"{idx}: {item}")
                processings.append(processing_syn_mapping[item])
                # print(processings, "\n\n")
                if processing_syn_mapping[item] not in processings_list:
                    processings_list.append(processing_syn_mapping[item])
            else:
                processings.append(item)
                if item not in processings_list:
                    processings_list.append(item)
                    
        output['PROCESSING'] = [i for i in processings if i not in ignore_processing]

    if output['FILLER']:
        for filler_item in output['FILLER']:
            if 'filler_name' in filler_item:
                fillers = []
                for item in filler_item['filler_name']:
                    item = item.lower()
                    if item in list(filler_syn_mapping):
                        # print(f"{idx}: {item}")
                        fillers.append(filler_syn_mapping[item])
                        # print(fillers, "\n\n")
                        if filler_syn_mapping[item] not in fillers_list:
                            fillers_list.append(filler_syn_mapping[item])
                    else:
                        fillers.append(item)
                        if item not in fillers_list:
                            fillers_list.append(item)
                            
                filler_item['filler_name'] = fillers

    if output['PROPERTY']:
        for prop_item in output['PROPERTY']:
            if 'property_name' in prop_item:
                if prop_item['property_name'].strip()=='':
                    print(row['Ref No.'], "PROPERTY NAME EMPTY STRING")
                    
                if prop_item['property_name'] in list(prop_syn_mapping):
                    # print(f"{idx}: {item}")
                    prop_item['property_name'] = prop_syn_mapping[prop_item['property_name']]
                    # print(properties, "\n\n")
                    if prop_item['property_name'] not in properties_list:
                        properties_list.append(prop_item['property_name'])
                else:
                    if prop_item['property_name'] not in properties_list:
                        properties_list.append(prop_item['property_name'])
                        
            if 'modifier' in prop_item:
                if prop_item['modifier']['value']:
                    try:
                        float(prop_item['modifier']['value'])
                    except:
                        if prop_item['modifier']['value'] in cat_values_mapping:
                            # print(prop_item['modifier']['value'])
                            prop_item['modifier']['value'] = cat_values_mapping[prop_item['modifier']['value']]
                            
                        else:
                            if prop_item['modifier']['value'] not in cat_values_list:
                                cat_values_list.append(prop_item['modifier']['value'])
                        
                        # print(output, "\n")
                else:
                    prop_item['modifier']['unit'] = ''

            # if prop_item['property_name']=='flame rating' and not prop_item['modifier']['value']:
            #     prop_item['modifier']['value'] = 'v-0' # do not add default value as you dont need for yellow card, ul listing etc

        # remove only after updating property dict
        remove_property = []
        for prop_item in output['PROPERTY']:
            if 'property_name' in prop_item and prop_item['property_name'] in ignore_property:
                print("removing property:", prop_item)
                print(row['Ref No.'])
                remove_property.append(prop_item)
        
        output['PROPERTY'] = [i for i in output['PROPERTY'] if i not in remove_property]

    if output['AUTO_CERT']:
        for c in output['AUTO_CERT']:
            if c['oem'] in auto_syns_mapping:
                c['oem'] = auto_syns_mapping[c['oem']]
            else:
                if c['oem'] not in oems_list:
                    oems_list.append(c['oem'])

    if output['APPLICATION']:
        for app in ignore_query_with_apps:
            if app in output['APPLICATION']:
                ignore_rows.append(row['Ref No.'])
                
        apps = []
        for app in output['APPLICATION']:
            apps.extend(app.split(';'))

        output['APPLICATION'] = [re.sub("(\s+)", " ", i.strip()) for i in apps]

    if output['NSF_CERT']:
        nsf_certs_labelled = []
        for x in output['NSF_CERT']:
            if x == '61':
                nsf_certs_labelled.append('nsf 61')
            else:
                nsf_certs_labelled.append(x)

        output['NSF_CERT'] = nsf_certs_labelled

    if output['RAILWAY_CERT']:
        rail_certs_labelled = []
        for x in output['RAILWAY_CERT']:
            if x['standard'] == 'en45545-2':
                x['standard'] = 'en 45545-2'

            rail_certs_labelled.append(x)
        
        output['RAILWAY_CERT'] = rail_certs_labelled

    for field in output:
        temp = []
        [temp.append(item) for item in output[field] if item not in temp]
        output[field] = temp
    
        
    # except exception as e:
    #     print(e)
    #     print(row['Ref No.'], output, "\n\n")
        
    outputs.append(output)
                
df["Output"] = outputs
df

removing property: {'property_name': 'shear', 'modifier': {'value': '453.9', 'min': '408.51', 'max': '499.29', 'unit': 'mpa'}, 'property_type': 'property'}
labeled_data_survey_queries 2_filtered_357
removing property: {'property_name': 'pellet size', 'modifier': {'value': '40.0', 'min': '36.0', 'max': '44.0', 'unit': 'pellets/g'}, 'property_type': 'property'}
labeled_data_survey_queries 2_filtered_361
removing property: {'property_name': 'shear', 'modifier': {'value': '453.9', 'min': '408.51', 'max': '499.29', 'unit': 'mpa'}, 'property_type': 'property'}
labeled_data_survey_queries 2_filtered_1927
removing property: {'property_name': 'crystallinity', 'modifier': {'value': '30.0', 'min': '30.0', 'max': '100.0', 'unit': '%'}, 'property_type': 'property'}
labeled_data_survey_queries 2_filtered_1965
removing property: {'property_name': 'pellet size', 'modifier': {'value': '40.0', 'min': '36.0', 'max': '44.0', 'unit': 'pellets / g'}, 'property_type': 'property'}
labeled_data_survey_queries 

,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5
...,...,...,...,...
61527,61528,what is the glass transition of fp 6e5901a80?,"{'GRADE': ['fp 6e5901a80'], 'APPLICATION': [],...",prod_generated_data_prop_grade_27_01_25_46
61528,61529,elongational stress f of lfrt cfr-tp pet gf60-10,"{'GRADE': ['lfrt cfr-tp pet gf60-10'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_47
61529,61530,tensile creep modulus at 1h of frianyl a3 v2 o...,"{'GRADE': ['frianyl a3 v2 or 2003/p'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_48
61530,61531,flexural stress at 3.5% iso 178 (mpa) of impet...,"{'GRADE': ['impet 830r'], 'APPLICATION': [], '...",prod_generated_data_prop_grade_27_01_25_49


In [18]:
ignore_rows

[]

In [19]:
df = df[~df['Ref No.'].isin(ignore_rows)]

In [20]:
# for idx, row in df[df['Ref No.'].isin(ignore_rows)].iterrows():
#     print(row['Query'])
#     output = row['Output']
#     for col in output:
#         if output[col]:
#             print(col, ':', output[col])
#     print( "\n\n")

In [21]:
df["Sl. No."] = list(range(1, len(df)+1))
df.to_excel("data/Training_Data_27_01_25/Training_Data_27_01_25.xlsx", header=True, index=False)